In [1]:
import scipy.io
import networkx as nx 
import bct 
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import nilearn
from nilearn import datasets, plotting, surface

# Preparation of dataset for use
number_subjects = 9

# definition of directory with dataset
dir_fmri_desikan = '/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_desikan/'
dir_fmri_destrieux = '/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_destrieux/'

dir_eeg_desikan = '/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/'
dir_eeg_destrieux = '/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_destrieux/'

# importing labels of atlas
mat_desikan = scipy.io.loadmat('/strombolihome/fribeiro/Dataset/source_reconstructed_FC/label_dsk.mat', squeeze_me=True)
labels_desikan = mat_desikan['label_dsk']
labels_desikan[67] = 'rINS' # small correction

mat_destrieux = scipy.io.loadmat('/strombolihome/fribeiro/Dataset/source_reconstructed_FC/label_dstrx.mat', squeeze_me=True)
labels_destrieux = mat_destrieux['label_dstrx']
labels_destrieux = labels_destrieux[12:160]
labels_destrieux[147] = 'rS_temporal_transverse' # small correction


In [2]:
# Script for creating graphs from fMRI and EEG connectivity data (coverting MATLAB matrices)

#1.Creation of average Graph 
def createAvGraph(file,data_type):
    
    # Load Matlab file with connectivity matrix for each time point - 3D matrix
    mat = scipy.io.loadmat(file)

    if data_type == 'fmri':
        conn_matrix = mat['connFMRI'].transpose() #connectivity fMRI matrix with numpy format
    elif data_type == 'eeg': #for now only for broad band TODO others !
        conn_matrix = mat['connEEGbroad'].transpose() #connectivity EEG matrix with numpy format
    elif data_type == 'eeg_alpha':
        conn_matrix = mat['connEEGalpha'].transpose()
    elif data_type == 'eeg_beta':
        conn_matrix = mat['connEEGbeta'].transpose()
    elif data_type == 'eeg_delta':
        conn_matrix = mat['connEEGdelta'].transpose()
    elif data_type == 'eeg_gamma':
        conn_matrix = mat['connEEGgamma'].transpose()
    elif data_type == 'eeg_theta':
        conn_matrix = mat['connEEGtheta'].transpose()
        
    t_points = conn_matrix.shape[0] # number of layers of multilayer matrix
    num_areas = conn_matrix.shape[1] #number of nodes of the graph
    
    # Get average connectivity matrix
    av_conn = np.zeros((num_areas,num_areas))
       
    for i in range(t_points):
        
        av_conn = av_conn + conn_matrix[i]
            
    av_conn = av_conn/t_points 
    
    G = nx.from_numpy_matrix(av_conn)
    
    return G

#2. Creation of array of graphs (equivalent to layers)
def createArrayGraph(file,data_type):
    
    # Load Matlab file with connectivity matrix for each time point - 3D matrix
    mat = scipy.io.loadmat(file)
    
    if data_type == 'fmri':
        conn_matrix = mat['connFMRI'].transpose() #connectivity fMRI matrix with numpy format
    elif data_type == 'eeg': #for now only for broad band TODO others !
        conn_matrix = mat['connEEGbroad'].transpose() #connectivity EEG matrix with numpy format
    elif data_type == 'eeg_alpha':
        conn_matrix = mat['connEEGalpha'].transpose()
    elif data_type == 'eeg_beta':
        conn_matrix = mat['connEEGbeta'].transpose()
    elif data_type == 'eeg_delta':
        conn_matrix = mat['connEEGdelta'].transpose()
    elif data_type == 'eeg_gamma':
        conn_matrix = mat['connEEGgamma'].transpose()
    elif data_type == 'eeg_theta':
        conn_matrix = mat['connEEGtheta'].transpose()

    t_points = conn_matrix.shape[0] # number of layers of multilayer matrix
    num_areas = conn_matrix.shape[1] #number of nodes of the graph
    
    array_graphs = np.empty(t_points, dtype=object) 
    
    for i in range(t_points):
        array_graphs[i] = nx.from_numpy_matrix(conn_matrix[i])
        
    return array_graphs

#3. Threshold graph with given proportion
def thresholdGraph(G, threshold, type_data):
    
    if(type_data == 'fmri'):
        # get absolute value connectivity matrix
        conn_matrix = abs(nx.to_numpy_array(G))
    else:
        conn_matrix = nx.to_numpy_array(G)
        
    # to threshold graph keeping top X% of the edges
    conn_new = bct.utils.threshold_proportional(conn_matrix, threshold, True)
    
    #for the fMRI data we need to recover non-absolute values as Phase Coherence is between -1 and 1
    if(type_data == 'fmri'):
        
        ind_keep = np.transpose(np.nonzero(conn_new))
        conn_new = np.zeros((G.number_of_nodes(), G.number_of_nodes()))
        min_positive = 1 #to store minimum positive value kept of phase coherence
        max_negative = -1 #to store maximum negative value kept of phase coherence
        
        for ind in ind_keep:
            #print(nx.to_numpy_array(G)[ind[0],ind[1]])
            conn_new[ind[0],ind[1]] = nx.to_numpy_array(G)[ind[0],ind[1]]
            #print(conn_new[ind[0],ind[1]])
            if nx.to_numpy_array(G)[ind[0],ind[1]] > 0 and nx.to_numpy_array(G)[ind[0],ind[1]] < min_positive:
                min_positive = nx.to_numpy_array(G)[ind[0],ind[1]]
            elif nx.to_numpy_array(G)[ind[0],ind[1]] < 0 and nx.to_numpy_array(G)[ind[0],ind[1]] > max_negative:
                max_negative = nx.to_numpy_array(G)[ind[0],ind[1]]
        
        G_new = nx.from_numpy_matrix(conn_new)
        
        return [G_new, conn_new, min_positive, max_negative]
    
    else:
        min_coh = np.amin(conn_new) #to store minimum value kept of imaginary part of coherency
        G_new = nx.from_numpy_matrix(conn_new)
        
        return [G_new, conn_new, min_coh] 
    

In [5]:
#4.Find minimum threshold that keeps giant component (hold at least 90% of the graph's nodes) - fMRI data

# Function to obtain the components of a graph plotting the giant component and the second biggest component
def getSizeComponents(G):
    
    graph_components = sorted(nx.connected_components(G), key=len, reverse=True)
        
    giant = G.subgraph(graph_components[0]).copy() #giant component for this threshold
    #nx.draw_networkx(giant)
    #plt.figure(figsize=(100, 100))
    #plt.show()
    print("Number of nodes of the giant component:", giant.number_of_nodes())
        
    if len(graph_components) > 1 :
        second = G.subgraph(graph_components[1]).copy()
        #nx.draw_networkx(second)
        #plt.figure(figsize=(50, 50))
        #plt.show()
        print("Number of nodes of the second biggest component:", second.number_of_nodes())
        
    return graph_components
              
    
# Function to compare giant component and the second biggest component, varying the threshold used
def findThresholdGiantComponent(G, proportion_values):
    
    # get absolute value connectivity matrix
    conn_matrix = abs(nx.to_numpy_array(G))
     
    # to obtain connectivity matrix thresholded and the corresponding "giant" component
    for p in proportion_values:
        
        print("Proportion kept:", p*100)
        new = bct.utils.threshold_proportional(conn_matrix, p, True) # option to use a proportional threshold - BCT
        G = nx.from_numpy_matrix(new)
        nodes = G.number_of_nodes()
        edges = G.number_of_edges()
        print("Number of edges of thresholded graph:", edges)
        av_degree = (2*edges)/nodes
        print("Average degree of thresholded graph: ", av_degree)
        
        graph_components = getSizeComponents(G)
        
        print("___________________________________________________________________________")
        
    # return value only matters to compare giant component    
    if len(graph_components) > 1:
        return ((G.subgraph(graph_components[0]).copy()).number_of_nodes() - (G.subgraph(graph_components[1]).copy()).number_of_nodes())
    else:
        return 0
    


In [6]:
#5.Check giant component for each time point of each subject - fMRI data and EEG data

def checkGiantComponent(G_array, proportion_values, atlas):
    
    array_problem_time_points = np.zeros(len(G_array))
    t = 0
    
    #arbitrary difference defined between giant and second biggest component
    if atlas == 'dsk':
        limit = 15 
    elif atlas == 'dstrx':
        limit = 30
    
    for G in G_array:
        t +=  1
        print("Graph for time point", t)
        
        difference = findThresholdGiantComponent(G,proportion_values)
            
        if difference < limit and difference != 0:
                array_problem_time_points[t-1] = t
           
    return array_problem_time_points

In [3]:
#6. Check size distribution of graph components for each time point - fMRI and EEG data

#to obtain graph components already with threshold graph
def getThresholdComponents(G,threshold,data):
    
    if data == 'fmri':
        G, conn_matrix, min_pos, max_neg = thresholdGraph(G, threshold, data)
    else:
        G, conn_matrix, min_coh = thresholdGraph(G, threshold, data)
        
    graph_components = sorted(nx.connected_components(G), key=len, reverse=True)
    
    return graph_components
    
    
# plotting size distribution of graph components
def distComponents(G_array,threshold,data,atlas,subject):
    
    time_point = 0
    max_components = 150 #arbitrary choice to be sure to include all components
    
    size_components = np.zeros((max_components,len(G_array)))
    
    for G in G_array:
        time_point += 1
        #print("Components for time frame: ", time_point)
        
        graph_components = getThresholdComponents(G,threshold[0],data)
        #print((G.subgraph(graph_components[0]).copy()).number_of_nodes())
        
        for i in range(0,len(graph_components)):
            nodes = (G.subgraph(graph_components[i]).copy()).number_of_nodes()
            size_components[i][(time_point-1)] = nodes
   
    print(size_components)
    # plotting the number of nodes, i.e, size of each component
    plt.figure(figsize=(42, 12))
    
    for j in range(0,max_components):
        if j == 0:
            plt.bar(np.arange(1,len(G_array)+1), size_components[j], align='edge', width=0.5, tick_label = np.arange(1,len(G_array)+1), alpha=0.5)
            bottom_bar = np.array(size_components[j])
        else:
            plt.bar(np.arange(1,len(G_array)+1), size_components[j], align='edge', width=0.5, bottom = bottom_bar, tick_label = np.arange(1,len(G_array)+1), alpha=0.5)
            bottom_bar += np.array(size_components[j])
            
    plt.ylabel("Number of nodes")
    plt.xlabel("Components")
    plt.title("Distribution of the components' size")
        
        
    plt.savefig('/strombolihome/fribeiro/Results/distribution_components/dist_components_subj' + str(subject) + '_threshold' + str(threshold[0]*100) + '_' + data + '_' + atlas + '.png')
    
    plt.show()
    

In [3]:
#7. Function to obtain node three-dimensional coordinates for both atlas

#to obtain three dimensional coordinates
def getNodeCoordinates(atlas):
    
    # using the nilearn package - code adapted from nilearn Github examples
    if atlas == 'dstrx':
        
        destrieux_atlas = datasets.fetch_atlas_surf_destrieux()
        # to retrieve fsaverage5 surface dataset for the plotting background
        fsaverage = datasets.fetch_surf_fsaverage()
        
        coordinates = []
        labels = destrieux_atlas['labels'] #includes 76 labels per hemisphere (extras: unknown and Medial_wall)
        labels[42] = labels[0] #to switch Medial_wall to unknown so as not to be taken into account
            
        for hemi in ['left', 'right']:
            vert = destrieux_atlas['map_%s' % hemi]
            rr, _ = surface.load_surf_mesh(fsaverage['pial_%s' % hemi])
            for k, label in enumerate(labels):
                if "Unknown" not in str(label):  # to omit the Unknown label.
                    # compute mean location of vertices in label of index k
                    coordinates.append(np.mean(rr[vert == k], axis=0))

        coordinates = np.array(coordinates)  # 3D coordinates of parcels - (N,3)
        
        np.save('/strombolihome/fribeiro/Thesis_project/Results/coordinates_nodes_' + atlas + '.npy', coordinates)
        
        #return coordinates
    
    else:
        # for Desikan atlas - coordinates extracted from file from BrainNet viewer toolbox
        f = open('/strombolihome/fribeiro/Dataset/source_reconstructed_FC/desi_coordinates.txt', 'r+')
        file = [line for line in f.readlines()]
        file.pop(0)
        f.close()
        
        coordinates = []
        for i in range(0,len(file)): 
            node_coord = file[i].split()[:-3] #processing of the file to isolate coordinates and remove extra information
            node_coord = [float(idx) for idx in node_coord] #convert to float
            coordinates.append(node_coord)
        
        np.save('/strombolihome/fribeiro/Thesis_project/Results/coordinates_nodes_' + atlas + '.npy', coordinates)
        
        #return coordinates # 3D coordinates of parcels - (N,3)

In [3]:
#8. Plot giant component over brain (anatomically correct)

def visualizeGiantComponent(G_array, type_data, atlas, threshold, subject, mode):
    
    time_point = 0
    
    for G in G_array:
        
        time_point += 1
        
        print("Giant component for time frame: ", time_point)
        
        if type_data == 'fmri':
            G, conn_matrix, min_pos, max_neg = thresholdGraph(G, threshold[0], type_data)
        else:
            G, conn_matrix, min_coh = thresholdGraph(G, threshold[0], type_data)

        giant_component = G.subgraph(max(nx.connected_components(G), key=len)).copy()
        #print(list(giant_component.nodes))

        # to remove edges of connectivity matrix not belonging to giant component
        for i in range(0,G.number_of_nodes()):
            if i not in list(giant_component.nodes):
                conn_matrix[i] = 0 #zeros out row i
                conn_matrix[:,i] = 0 #zeros out column i
        
        coordinates = np.load('/strombolihome/fribeiro/Thesis_project/Results/coordinates_nodes_' + atlas + '.npy')
        
        if mode == '2D':
            
            view = plotting.plot_connectome(conn_matrix, coordinates, title='Giant Component connectome', node_size = 10)
            #plotting.show()
            view.savefig('/strombolihome/fribeiro/Thesis_project/Results/plot_components/giant_component/subj0' + str(subject) + '/' + type_data + '/giant_component_t' + str(time_point) + '_' + mode + '_threshold' + str(threshold[0]*100) + '_' + atlas + '.png') 
            view.close() 
            
        elif mode == '3D':
            
            view = plotting.view_connectome(conn_matrix, coordinates, title = 'Giant Component Connectome')    
            view.save_as_html('/strombolihome/fribeiro/Thesis_project/Results/plot_components/giant_component/subj0' + str(subject) + '/' + type_data + '/giant_component_t' + str(time_point) + '_' + mode + '_threshold' + str(threshold[0]*100) + '_' + atlas + '.html')        
        

In [12]:
#9. Giant component's parcellates plotted over the brain

from visbrain.gui import Brain
from visbrain.objects import BrainObj, SceneObj, ColorbarObj
from visbrain.io import download_file

#plotting parcels belonging to giant component in each time frame
def visualizeROIsGiantComponent(G_array, type_data, atlas, threshold, subject):
    
    #definition of scene that will include all the plots
    sc = SceneObj(bgcolor='black', size=(39600, 3000))
    KW = dict(title_size=14., zoom=1.2)
    
    # get atlas file
    if atlas == 'dsk':
        file = 'rh.aparc.annot' #FreeSurfer Desikan Atlas filename
    else:
        file = 'lh.aparc.a2009s.annot' #FreeSurfer Destrieux Atlas filename
    
    path_to_file = download_file(file, astype='example_data')
   
    # create brain object to obtain the parcellation
    brain_obj = BrainObj('white', hemisphere='both', translucent=False,
                 cblabel='Brain representation of Giant Component regions', cbtxtsz=4.)

    df = brain_obj.get_parcellates(path_to_file)
    aux = df['Labels']
    parcels = [] # to store the parcellates belonging to each atlas
    
    # to remove unwanted parcels to have correspondence with giant component's nodes
    for i in range(0,len(aux)):
        if atlas == 'dsk':
            if aux[i] != 'corpuscallosum' and aux[i] != 'unknown':
                parcels.append(aux[i])
        else:
            if aux[i] != 'unknown':
                parcels.append(aux[i])
    
    time_point = 0
    col_time_point = 0
    row_point = 0
    
    for G in G_array:
        
        print("Giant component for time frame: ", time_point)
        
        if type_data == 'fmri':
            G, conn_matrix, min_pos, max_neg = thresholdGraph(G, threshold[0], type_data)
        else:
            G, conn_matrix, min_coh = thresholdGraph(G, threshold[0], type_data)

        giant_component = G.subgraph(max(nx.connected_components(G), key=len)).copy()

        nodes_giant = list(giant_component.nodes) #index of brain regions belonging to giant component
        # 0-33 (or 0-73) left hemisphere and 34-67 (or 74-147) right hemisphere
        
        select_parcels_left = []
        select_parcels_right = []
        # association between node index and brain region for each hemisphere
        for i in nodes_giant:
            if i < len(parcels):
                select_parcels_left.append(parcels[i])
            else:
                select_parcels_right.append(parcels[i-len(parcels)])
                
        brain_obj_G = BrainObj('white', hemisphere='both', translucent=False,
                 cblabel='Brain representation of Giant Component regions', cbtxtsz=4.)
        
        data_left = np.arange(len(select_parcels_left))
        brain_obj_G.parcellize(path_to_file, hemisphere='left', select=select_parcels_left, data=data_left,
                 cmap='Spectral_r')
        
        data_right = np.arange(len(select_parcels_right))
        brain_obj_G.parcellize(path_to_file, hemisphere='right', select=select_parcels_right, data=data_right,
                 cmap='Spectral_r')
        
        brain_obj_G.animate()
        
        sc.add_to_subplot(brain_obj_G, row=row_point, col=col_time_point, title='Giant Component t = ' + str(time_point),  rotate='right', **KW)
        
        time_point += 1
        row_point += 1
        
        #print("This:", time_point%10)
        
        if time_point%10 == 0:
            print("change of row")
            col_time_point += 1
            row_point = 0
     
    #output_name = '/strombolihome/fribeiro/Thesis_project/Results/plot_regions/giant_component/subj0' + str(subject) + '/' + type_data + '/regions_giant_component_t' + str(time_point) + '_threshold' + str(threshold[0]*100) + '_' + atlas + '.gif' #to save animation
    #print(output_name)

    sc.record_animation('test_smaller.gif', n_pic=75) #to save animation    

    #TODO fix - RuntimeError: FrameBuffer attachments are incomplete.


In [16]:
G_array = createArrayGraph(dir_fmri_desikan + 'subj01-7T/conn_desi_phase_coh_time_fmri.mat', 'fmri')

visualizeROIsGiantComponent(G_array[0:20], 'fmri', 'dsk', [0.1], 1)

Creation of a scene
File already dowloaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot).
BrainObj(name='white') created
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)


Giant component for time frame:  1


BrainObj(name='white') created
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : bankssts, caudalanteriorcingulate, caudalmiddlefrontal, cuneus, entorhinal, fusiform, inferiorparietal, lateraloccipital, lateralorbitofrontal, lingual, middletemporal, parahippocampal, paracentral, parsopercularis, parsorbitalis, parstriangularis, postcentral, posteriorcingulate, precentral, rostralmiddlefrontal, superiorparietal, supramarginal, insula
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : bankssts, caudalanteriorcingulate, cuneus, entorhinal, fusiform, inferiorparietal, inferiortemporal, isthmuscingulate, lateraloccipital, lingual, parahippocampal, paracentral, parsopercularis, parsorbitalis, parstriangularis, pericalcarine, postcentral, precentral, r

This: 1
Giant component for time frame:  2


BrainObj(name='white') created
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : bankssts, caudalanteriorcingulate, cuneus, entorhinal, fusiform, inferiorparietal, inferiortemporal, lateraloccipital, lateralorbitofrontal, lingual, middletemporal, parahippocampal, paracentral, parsopercularis, parsorbitalis, parstriangularis, postcentral, posteriorcingulate, precentral, rostralmiddlefrontal, superiorparietal, supramarginal, insula
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : bankssts, caudalmiddlefrontal, cuneus, entorhinal, fusiform, inferiorparietal, inferiortemporal, lateraloccipital, lingual, parahippocampal, paracentral, parsopercularis, parsorbitalis, parstriangularis, pericalcarine, postcentral, posteriorcingulate, precentral, rostra

...

BrainObj(name='white') created
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : bankssts, caudalanteriorcingulate, caudalmiddlefrontal, cuneus, entorhinal, fusiform, inferiorparietal, inferiortemporal, isthmuscingulate, lateraloccipital, lateralorbitofrontal, lingual, medialorbitofrontal, middletemporal, parahippocampal, paracentral, parsopercularis, parsorbitalis, parstriangularis, pericalcarine, postcentral, posteriorcingulate, precentral, precuneus, rostralanteriorcingulate, rostralmiddlefrontal, superiorfrontal, superiorparietal, superiortemporal, supramarginal, frontalpole, temporalpole, transversetemporal, insula
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : bankssts, caudalanteriorcingulate, caudalmiddlefrontal, cuneus, entorhinal, 

This: 9
Giant component for time frame:  20


BrainObj(name='white') created
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : bankssts, caudalanteriorcingulate, caudalmiddlefrontal, cuneus, entorhinal, fusiform, inferiorparietal, inferiortemporal, isthmuscingulate, lateraloccipital, lateralorbitofrontal, lingual, medialorbitofrontal, middletemporal, parahippocampal, paracentral, parsopercularis, parsorbitalis, parstriangularis, pericalcarine, postcentral, posteriorcingulate, precentral, precuneus, rostralanteriorcingulate, rostralmiddlefrontal, superiorfrontal, superiorparietal, superiortemporal, supramarginal, frontalpole, temporalpole, transversetemporal, insula
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : bankssts, caudalanteriorcingulate, caudalmiddlefrontal, cuneus, entorhinal, 

This: 0
change of row


RuntimeError: FrameBuffer attachments are incomplete.

In [6]:
#10. Comparison partitions/regions of giant component between time frames - NMI

# Metrics
# NMI: 0 - no mutual information; 1 - perfect correlation

import igraph as ig
from igraph import *

def comparisonComponents(G_array, threshold, type_data, atlas, subject, mode):
    
    if mode == 'crash':
        nmi = np.load('/strombolihome/fribeiro/Thesis_project/Results/NMI_connected_components/NMI_components_' + str(subject) + '_threshold' + str(threshold[0]*100) + '_' + type_data + '_' + atlas + '.npy')
    else:   
        nmi = np.zeros((len(G_array), len(G_array))) #comparison between all time points component distribution
        np.fill_diagonal(nmi, 1) #fill the diagonal as NMI will be 1
    
    g = 0
    for G in G_array:
        
        graph_components = getThresholdComponents(G,threshold[0],type_data)
        
        list_comp =[0]*G.number_of_nodes()

        #prepare array with component label for each node
        i=0
        for comp in graph_components:
            for node in comp:
                list_comp[int(node)]=i
            i+=1
        
        aux = g + 1
        g_after = aux
        #to compare each component distribution with the original one (upper triangle only)
        for G_after in G_array[aux:]:
            
            print("start:", g_after)
            
            graph_components_after = getThresholdComponents(G_after,threshold[0],type_data)
            
            list_comp_after =[0]*G_after.number_of_nodes()

            j=0
            for comp in graph_components_after:
                for node in comp:
                    list_comp_after[int(node)]=j
                j+=1
            
            nmi[g,g_after] = compare_communities(list_comp, list_comp_after, method='nmi')
            nmi[g_after,g] = nmi[g,g_after] #matrix is symmetrical
            
            g_after +=1
        
        print("done:", g)
        g += 1
        #save if crashing
        np.save('/strombolihome/fribeiro/Thesis_project/Results/NMI_connected_components/NMI_components_' + str(subject) + '_threshold' + str(threshold[0]*100) + '_' + type_data + '_' + atlas + '.npy', nmi)
        
    print(nmi)
    
    np.save('/strombolihome/fribeiro/Thesis_project/Results/NMI_connected_components/NMI_components_' + str(subject) + '_threshold' + str(threshold[0]*100) + '_' + type_data + '_' + atlas + '.npy', nmi)
    
#11. Gruping the time points into classes according to NMI




In [ ]:
#12. Plotting component size distribution (# components with given size)

In [7]:
G_array = createArrayGraph(dir_fmri_desikan + 'subj01-7T/conn_desi_phase_coh_time_fmri.mat', 'fmri')

comparisonComponents(G_array, [0.1], 'fmri', 'dsk', 1)


done: 0
done: 1
done: 2
done: 3
done: 4
done: 5
done: 6
done: 7
done: 8
done: 9
done: 10
done: 11
done: 12
done: 13
done: 14
done: 15
done: 16
done: 17
done: 18
done: 19
done: 20
done: 21
done: 22
done: 23
done: 24
done: 25
done: 26
done: 27
done: 28
done: 29
done: 30
done: 31
done: 32
done: 33
done: 34
done: 35
done: 36


KeyboardInterrupt: 

In [ ]:
#13. Comparison partitions/regions of giant component between two modalities fMRI and EEG for all time frames - NMI

def comparisonComponentsModalities(G_array, threshold, type_data, atlas, subject):

In [4]:
#14. Component and threshold analysis functions for each subject, data type and atlas

def thresholdAnalysis(subject, type_data, atlas, file, threshold_values):
    
    print("Subject {}, with {} and using {}".format(subject,type_data,atlas))
    
    #print(file)
    
    G = createAvGraph(file, type_data)
    
    difference = findThresholdGiantComponent(G, threshold_values[atlas])

# to be run for a given threshold (in array shape)
def componentAnalysis(subject, type_data, atlas, file, threshold, mode):
    
    G_array = createArrayGraph(file, type_data)
    
    ############################################################################################################################################################################################
    # Check time points for which the giant component disappears (size similar to second biggest component) with a given threshold
    
    #problem_time_points = checkGiantComponent(G_array, threshold, atlas)
    #output_name = '/strombolihome/fribeiro/Results/problem_time_points/problem_time_points_subj' + str(subject) + '_threshold' + str(threshold[0]*100) + '_' + type_data + '_' + atlas + '.txt'
    #output = problem_time_points[np.nonzero(problem_time_points)]
    #print("These are the time frames for which the giant component was compromised, for subject {}, with {} and using {} atlas: {}".format(subject, type_data, atlas, output))
    #np.savetxt(output_name, output, delimiter=',', fmt = '%10d')
    
    ############################################################################################################################################################################################
    # Obtain the distribution of the components' size (number of nodes) for each time point
    
    #distComponents(G_array, threshold, type_data, atlas, subject)
    
    #############################################################################################################################################################################################
    # Plot giant component over brain representation - mode '2D' or '3D'
    
    visualizeGiantComponent(G_array, type_data, atlas, threshold, subject, mode)
    
    ##############################################################################################################################################################################################
    # Plot giant component regions according to atlas over brain representation
    
    #visualizeROIsGiantComponent(G_array, type_data, atlas, threshold, subject)

    

In [ ]:
# proportion of top edges to keep of the graph
proportion = {'dsk': [0.01, 0.015, 0.016, 0.018, 0.02, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.25, 0.5, 1, 1, 1, 1, 1], 'dstrx': [0.005, 0.008, 0.01, 0.012, 0.015, 0.018, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.25, 0.5, 1]}

threshold_values = pd.DataFrame(data = proportion)

print(threshold_values)


In [22]:
# Desikan atlas - fMRI

s = 1

for subdir, dirs, files in sorted(os.walk(dir_fmri_desikan)):
    
    for file in files:
        
        if 'conn_desi_phase_coh' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'fmri','dsk',os.path.join(subdir, file), threshold_values)
            s += 1


Threshold analysis for subject 1, with fmri and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_desikan/subj01-7T/conn_desi_phase_coh_time_fmri.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 6
Number of nodes of the second biggest component: 6
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 9
Number of nodes of the second biggest component: 8
___________________________________________________________________________
Proportion kept: 1.6
Number of edges of thresholded graph: 36
Average degree of thresholded graph:  1.0588235294117647
Number of nodes of the giant component: 9
Number of nodes of the second biggest component: 9
________________________________________________

Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 3, with fmri and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_desikan/subj03-7T/conn_desi_phase_coh_time_fmri.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 11
Number of nodes of the second biggest component: 4
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph: 

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 5, with fmri and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_desikan/subj05-7T/conn_desi_phase_coh_time_fmri.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 7
Number of nodes of the second biggest component: 5
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 8
Number of nodes of the second biggest co

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 7, with fmri and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_desikan/subj07-7T/conn_desi_phase_coh_time_fmri.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 7
Number of nodes of the second biggest component: 6
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 19
Number of nodes of the second biggest c

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 9, with fmri and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_desikan/subj09-7T/conn_desi_phase_coh_time_fmri.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 9
Number of nodes of the second biggest component: 3
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 13
Number of nodes of the second biggest c

In [ ]:
# Desikan atlas - fMRI

#componentAnalysis(1,'fmri','dsk', dir_fmri_desikan + 'subj01-7T/conn_desi_phase_coh_time_fmri.mat', [0.1], '2D')
#componentAnalysis(2,'fmri','dsk', dir_fmri_desikan + 'subj02-7T/conn_desi_phase_coh_time_fmri.mat', [0.1], '2D')
#componentAnalysis(3,'fmri','dsk', dir_fmri_desikan + 'subj03-7T/conn_desi_phase_coh_time_fmri.mat', [0.1], '2D')
#componentAnalysis(4,'fmri','dsk', dir_fmri_desikan + 'subj04-7T/conn_desi_phase_coh_time_fmri.mat', [0.1], '2D')
#componentAnalysis(5,'fmri','dsk', dir_fmri_desikan + 'subj05-7T/conn_desi_phase_coh_time_fmri.mat', [0.1], '2D')
#componentAnalysis(6,'fmri','dsk', dir_fmri_desikan + 'subj06-7T/conn_desi_phase_coh_time_fmri.mat', [0.1], '2D')
#componentAnalysis(7,'fmri','dsk', dir_fmri_desikan + 'subj07-7T/conn_desi_phase_coh_time_fmri.mat', [0.1], '2D')
#componentAnalysis(8,'fmri','dsk', dir_fmri_desikan + 'subj08-7T/conn_desi_phase_coh_time_fmri.mat', [0.1], '2D')
#componentAnalysis(9,'fmri','dsk', dir_fmri_desikan + 'subj09-7T/conn_desi_phase_coh_time_fmri.mat', [0.1], '2D')

#componentAnalysis(1,'fmri','dsk', dir_fmri_desikan + 'subj01-7T/conn_desi_phase_coh_time_fmri.mat', [0.1], '3D')
#componentAnalysis(2,'fmri','dsk', dir_fmri_desikan + 'subj02-7T/conn_desi_phase_coh_time_fmri.mat', [0.1], '3D')
#componentAnalysis(3,'fmri','dsk', dir_fmri_desikan + 'subj03-7T/conn_desi_phase_coh_time_fmri.mat', [0.1], '3D')
#componentAnalysis(4,'fmri','dsk', dir_fmri_desikan + 'subj04-7T/conn_desi_phase_coh_time_fmri.mat', [0.1], '3D')
#componentAnalysis(5,'fmri','dsk', dir_fmri_desikan + 'subj05-7T/conn_desi_phase_coh_time_fmri.mat', [0.1], '3D')
#componentAnalysis(6,'fmri','dsk', dir_fmri_desikan + 'subj06-7T/conn_desi_phase_coh_time_fmri.mat', [0.1], '3D')
#componentAnalysis(7,'fmri','dsk', dir_fmri_desikan + 'subj07-7T/conn_desi_phase_coh_time_fmri.mat', [0.1], '3D')
#componentAnalysis(8,'fmri','dsk', dir_fmri_desikan + 'subj08-7T/conn_desi_phase_coh_time_fmri.mat', [0.1], '3D')
#componentAnalysis(9,'fmri','dsk', dir_fmri_desikan + 'subj09-7T/conn_desi_phase_coh_time_fmri.mat', [0.1], '3D')

Creation of a scene
File already dowloaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot).
BrainObj(name='white') created
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)


Giant component for time frame:  1


BrainObj(name='white') created
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : bankssts, caudalanteriorcingulate, caudalmiddlefrontal, cuneus, entorhinal, fusiform, inferiorparietal, lateraloccipital, lateralorbitofrontal, lingual, middletemporal, parahippocampal, paracentral, parsopercularis, parsorbitalis, parstriangularis, postcentral, posteriorcingulate, precentral, rostralmiddlefrontal, superiorparietal, supramarginal, insula
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : bankssts, caudalanteriorcingulate, cuneus, entorhinal, fusiform, inferiorparietal, inferiortemporal, isthmuscingulate, lateraloccipital, lingual, parahippocampal, paracentral, parsopercularis, parsorbitalis, parstriangularis, pericalcarine, postcentral, precentral, r

Giant component for time frame:  2


BrainObj(name='white') created
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : bankssts, caudalanteriorcingulate, cuneus, entorhinal, fusiform, inferiorparietal, inferiortemporal, lateraloccipital, lateralorbitofrontal, lingual, middletemporal, parahippocampal, paracentral, parsopercularis, parsorbitalis, parstriangularis, postcentral, posteriorcingulate, precentral, rostralmiddlefrontal, superiorparietal, supramarginal, insula
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : bankssts, caudalmiddlefrontal, cuneus, entorhinal, fusiform, inferiorparietal, inferiortemporal, lateraloccipital, lingual, parahippocampal, paracentral, parsopercularis, parsorbitalis, parstriangularis, pericalcarine, postcentral, posteriorcingulate, precentral, rostra

...

BrainObj(name='white') created
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : caudalanteriorcingulate, caudalmiddlefrontal, fusiform, inferiortemporal, isthmuscingulate, lateraloccipital, lingual, parsopercularis, pericalcarine, postcentral, posteriorcingulate, precuneus, superiorfrontal, superiorparietal, supramarginal
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : bankssts, caudalanteriorcingulate, caudalmiddlefrontal, entorhinal, fusiform, inferiorparietal, inferiortemporal, isthmuscingulate, lateralorbitofrontal, parsopercularis, parsorbitalis, parstriangularis, pericalcarine, posteriorcingulate, precentral, precuneus, superiorfrontal, superiorparietal, supramarginal
    BrainObj(name='white') added to the scene


Giant component for time frame:  399


BrainObj(name='white') created
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : bankssts, caudalanteriorcingulate, caudalmiddlefrontal, cuneus, entorhinal, fusiform, inferiorparietal, inferiortemporal, isthmuscingulate, lateraloccipital, lateralorbitofrontal, lingual, paracentral, parsopercularis, parstriangularis, pericalcarine, postcentral, posteriorcingulate, precuneus, rostralanteriorcingulate, superiorfrontal, superiorparietal, superiortemporal, supramarginal, temporalpole
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : bankssts, caudalanteriorcingulate, caudalmiddlefrontal, cuneus, entorhinal, fusiform, inferiorparietal, inferiortemporal, isthmuscingulate, lateraloccipital, lateralorbitofrontal, lingual, medialorbitofrontal, middletemp

Giant component for time frame:  400


BrainObj(name='white') created
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : bankssts, caudalanteriorcingulate, caudalmiddlefrontal, cuneus, entorhinal, fusiform, inferiorparietal, inferiortemporal, isthmuscingulate, lateraloccipital, lateralorbitofrontal, lingual, parahippocampal, paracentral, parsopercularis, parsorbitalis, parstriangularis, pericalcarine, postcentral, posteriorcingulate, precentral, precuneus, rostralanteriorcingulate, superiorfrontal, superiorparietal, superiortemporal, supramarginal, temporalpole
    Annot file loaded (/home/fribeiro/visbrain_data/example_data/rh.aparc.annot)
    Color inferred from data
    Search parcellates using labels
    Selected parcellates : bankssts, caudalanteriorcingulate, caudalmiddlefrontal, cuneus, entorhinal, fusiform, inferiorparietal, inferiortemporal, isthmuscingulate, lateraloccipital, lateralorbitofront

In [23]:
# Destrieux atlas - fMRI

s = 1

for subdir, dirs, files in sorted(os.walk(dir_fmri_destrieux)):
    
    for file in files:
        
        if 'conn_destrx_phase_coh' in (os.path.join(subdir,file)):
    
            thresholdAnalysis(s,'fmri','dstrx',os.path.join(subdir, file), threshold_values)
            s += 1
        

Threshold analysis for subject 1, with fmri and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_destrieux/subj01-7T/conn_destrx_phase_coh_time_fmri.mat
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 16
Number of nodes of the second biggest component: 8
___________________________________________________________________________
Proportion kept: 0.8
Number of edges of thresholded graph: 87
Average degree of thresholded graph:  1.1756756756756757
Number of nodes of the giant component: 31
Number of nodes of the second biggest component: 9
___________________________________________________________________________
Proportion kept: 1.0
Number of edges of thresholded graph: 109
Average degree of thresholded graph:  1.472972972972973
Number of nodes of the giant component: 54
Number of nodes of the second biggest component: 9
________________________

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Threshold analysis for subject 3, with fmri and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_destrieux/subj03-7T/conn_destrx_phase_coh_time_fmri.mat
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Threshold analysis for subject 5, with fmri and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_destrieux/subj05-7T/conn_destrx_phase_coh_time_fmri.mat
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Threshold analysis for subject 7, with fmri and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_destrieux/subj07-7T/conn_destrx_phase_coh_time_fmri.mat
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Threshold analysis for subject 9, with fmri and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_destrieux/subj09-7T/conn_destrx_phase_coh_time_fmri.mat
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of

In [5]:
# Destrieux atlas - fMRI

#componentAnalysis(1,'fmri','dstrx', dir_fmri_destrieux + 'subj01-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06], '2D')
#componentAnalysis(2,'fmri','dstrx', dir_fmri_destrieux + 'subj02-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06], '2D')
#componentAnalysis(3,'fmri','dstrx', dir_fmri_destrieux + 'subj03-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06], '2D')
#componentAnalysis(4,'fmri','dstrx', dir_fmri_destrieux + 'subj04-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06], '2D')
#componentAnalysis(5,'fmri','dstrx', dir_fmri_destrieux + 'subj05-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06], '2D')
#componentAnalysis(6,'fmri','dstrx', dir_fmri_destrieux + 'subj06-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06], '2D')
#componentAnalysis(7,'fmri','dstrx', dir_fmri_destrieux + 'subj07-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06], '2D')
#componentAnalysis(8,'fmri','dstrx', dir_fmri_destrieux + 'subj08-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06], '2D')
componentAnalysis(9,'fmri','dstrx', dir_fmri_destrieux + 'subj09-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06], '2D')

#componentAnalysis(1,'fmri','dstrx', dir_fmri_destrieux + 'subj01-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06], '3D')
#componentAnalysis(2,'fmri','dstrx', dir_fmri_destrieux + 'subj02-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06], '3D')
#componentAnalysis(3,'fmri','dstrx', dir_fmri_destrieux + 'subj03-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06], '3D')
#componentAnalysis(4,'fmri','dstrx', dir_fmri_destrieux + 'subj04-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06], '3D')
#componentAnalysis(5,'fmri','dstrx', dir_fmri_destrieux + 'subj05-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06], '3D')
#componentAnalysis(6,'fmri','dstrx', dir_fmri_destrieux + 'subj06-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06], '3D')
#componentAnalysis(7,'fmri','dstrx', dir_fmri_destrieux + 'subj07-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06], '3D')
#componentAnalysis(8,'fmri','dstrx', dir_fmri_destrieux + 'subj08-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06], '3D')
#componentAnalysis(9,'fmri','dstrx', dir_fmri_destrieux + 'subj09-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06], '3D')

Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant component for time frame:  7
Giant component for time frame:  8
Giant component for time frame:  9
Giant component for time frame:  10
Giant component for time frame:  11
Giant component for time frame:  12
Giant component for time frame:  13
Giant component for time frame:  14
Giant component for time frame:  15
Giant component for time frame:  16
Giant component for time frame:  17
Giant component for time frame:  18
Giant component for time frame:  19
Giant component for time frame:  20
Giant component for time frame:  21
Giant component for time frame:  22
Giant component for time frame:  23
Giant component for time frame:  24
Giant component for time frame:  25
Giant component for time frame:  26
Giant component for time frame:  27
Giant component for time frame:  28
G

Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
Giant component for time frame:  232
Giant component for time frame:  233
Giant component for time frame:  234
Giant component for time frame:  235
Giant component for time frame:  236
Giant component for time frame:  237
Giant component for time frame:  238
Giant component for time frame:  239
Giant component for time frame:  240
Giant component for time frame:  241
Giant component for time frame:  242
Giant component for time frame:  243
Giant component for time frame:  244
Giant component for time frame:  245
Giant component for time frame:  246
Giant component for time frame:  247
Giant component for time frame:  248
Giant component for time frame:  249
Giant component for time frame:  250
Giant component for time frame:  251
Giant component for time frame:  252
G

In [27]:
# Desikan atlas - EEG broad band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_desikan)):
    
    for file in files:
        
        if 'conn_desi_cohi_time_eeg_broad' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg','dsk',os.path.join(subdir, file), threshold_values)
            s += 1


Threshold analysis for subject 1, with eeg and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj01-7T/conn_desi_cohi_time_eeg_broad_subj1-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 7
Number of nodes of the second biggest component: 4
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 12
Number of nodes of the second biggest component: 6
___________________________________________________________________________
Proportion kept: 1.6
Number of edges of thresholded graph: 36
Average degree of thresholded graph:  1.0588235294117647
Number of nodes of the giant component: 12
Number of nodes of the second biggest component: 6
______________________________________

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 3, with eeg and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj03-7T/conn_desi_cohi_time_eeg_broad_subj3-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 11
Number of nodes of the second biggest component: 4
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 14
Number of nodes of the second 

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 5, with eeg and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj05-7T/conn_desi_cohi_time_eeg_broad_subj5-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 6
Number of nodes of the second biggest component: 5
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 9
Number of nodes of the second bi

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 7, with eeg and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj07-7T/conn_desi_cohi_time_eeg_broad_subj7-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 11
Number of nodes of the second biggest component: 5
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 12
Number of nodes of the second 

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 9, with eeg and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj09-7T/conn_desi_cohi_time_eeg_broad_subj9-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 5
Number of nodes of the second biggest component: 3
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 9
Number of nodes of the second bi

In [5]:
# Desikan atlas - EEG broad band

#componentAnalysis(1,'eeg','dsk', dir_eeg_desikan + 'subj01-7T/conn_desi_cohi_time_eeg_broad_subj1-7T_.mat', [0.03], '2D')
#componentAnalysis(2,'eeg','dsk', dir_eeg_desikan + 'subj02-7T/conn_desi_cohi_time_eeg_broad_subj2-7T_.mat', [0.03], '2D')
#componentAnalysis(3,'eeg','dsk', dir_eeg_desikan + 'subj03-7T/conn_desi_cohi_time_eeg_broad_subj3-7T_.mat', [0.03], '2D')
#componentAnalysis(4,'eeg','dsk', dir_eeg_desikan + 'subj04-7T/conn_desi_cohi_time_eeg_broad_subj4-7T_.mat', [0.03], '2D')
#componentAnalysis(5,'eeg','dsk', dir_eeg_desikan + 'subj05-7T/conn_desi_cohi_time_eeg_broad_subj5-7T_.mat', [0.03], '2D')
#componentAnalysis(6,'eeg','dsk', dir_eeg_desikan + 'subj06-7T/conn_desi_cohi_time_eeg_broad_subj6-7T_.mat', [0.03], '2D')
#componentAnalysis(7,'eeg','dsk', dir_eeg_desikan + 'subj07-7T/conn_desi_cohi_time_eeg_broad_subj7-7T_.mat', [0.03], '2D')
componentAnalysis(8,'eeg','dsk', dir_eeg_desikan + 'subj08-7T/conn_desi_cohi_time_eeg_broad_subj8-7T_.mat', [0.03], '2D')
componentAnalysis(9,'eeg','dsk', dir_eeg_desikan + 'subj09-7T/conn_desi_cohi_time_eeg_broad_subj9-7T_.mat', [0.03], '2D')

#componentAnalysis(1,'eeg','dsk', dir_eeg_desikan + 'subj01-7T/conn_desi_cohi_time_eeg_broad_subj1-7T_.mat', [0.03], '3D')
#componentAnalysis(2,'eeg','dsk', dir_eeg_desikan + 'subj02-7T/conn_desi_cohi_time_eeg_broad_subj2-7T_.mat', [0.03], '3D')
#componentAnalysis(3,'eeg','dsk', dir_eeg_desikan + 'subj03-7T/conn_desi_cohi_time_eeg_broad_subj3-7T_.mat', [0.03], '3D')
#componentAnalysis(4,'eeg','dsk', dir_eeg_desikan + 'subj04-7T/conn_desi_cohi_time_eeg_broad_subj4-7T_.mat', [0.03], '3D')
#componentAnalysis(5,'eeg','dsk', dir_eeg_desikan + 'subj05-7T/conn_desi_cohi_time_eeg_broad_subj5-7T_.mat', [0.03], '3D')
#componentAnalysis(6,'eeg','dsk', dir_eeg_desikan + 'subj06-7T/conn_desi_cohi_time_eeg_broad_subj6-7T_.mat', [0.03], '3D')
#componentAnalysis(7,'eeg','dsk', dir_eeg_desikan + 'subj07-7T/conn_desi_cohi_time_eeg_broad_subj7-7T_.mat', [0.03], '3D')
#componentAnalysis(8,'eeg','dsk', dir_eeg_desikan + 'subj08-7T/conn_desi_cohi_time_eeg_broad_subj8-7T_.mat', [0.03], '3D')
#componentAnalysis(9,'eeg','dsk', dir_eeg_desikan + 'subj09-7T/conn_desi_cohi_time_eeg_broad_subj9-7T_.mat', [0.03], '3D')


Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant component for time frame:  7
Giant component for time frame:  8
Giant component for time frame:  9
Giant component for time frame:  10
Giant component for time frame:  11
Giant component for time frame:  12
Giant component for time frame:  13
Giant component for time frame:  14
Giant component for time frame:  15
Giant component for time frame:  16
Giant component for time frame:  17
Giant component for time frame:  18
Giant component for time frame:  19
Giant component for time frame:  20
Giant component for time frame:  21
Giant component for time frame:  22
Giant component for time frame:  23
Giant component for time frame:  24
Giant component for time frame:  25
Giant component for time frame:  26
Giant component for time frame:  27
Giant component for time frame:  28
G

Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
Giant component for time frame:  232
Giant component for time frame:  233
Giant component for time frame:  234
Giant component for time frame:  235
Giant component for time frame:  236
Giant component for time frame:  237
Giant component for time frame:  238
Giant component for time frame:  239
Giant component for time frame:  240
Giant component for time frame:  241
Giant component for time frame:  242
Giant component for time frame:  243
Giant component for time frame:  244
Giant component for time frame:  245
Giant component for time frame:  246
Giant component for time frame:  247
Giant component for time frame:  248
Giant component for time frame:  249
Giant component for time frame:  250
Giant component for time frame:  251
Giant component for time frame:  252
G

Giant component for time frame:  85
Giant component for time frame:  86
Giant component for time frame:  87
Giant component for time frame:  88
Giant component for time frame:  89
Giant component for time frame:  90
Giant component for time frame:  91
Giant component for time frame:  92
Giant component for time frame:  93
Giant component for time frame:  94
Giant component for time frame:  95
Giant component for time frame:  96
Giant component for time frame:  97
Giant component for time frame:  98
Giant component for time frame:  99
Giant component for time frame:  100
Giant component for time frame:  101
Giant component for time frame:  102
Giant component for time frame:  103
Giant component for time frame:  104
Giant component for time frame:  105
Giant component for time frame:  106
Giant component for time frame:  107
Giant component for time frame:  108
Giant component for time frame:  109
Giant component for time frame:  110
Giant component for time frame:  111
Giant component 

Giant component for time frame:  307
Giant component for time frame:  308
Giant component for time frame:  309
Giant component for time frame:  310
Giant component for time frame:  311
Giant component for time frame:  312
Giant component for time frame:  313
Giant component for time frame:  314
Giant component for time frame:  315
Giant component for time frame:  316
Giant component for time frame:  317
Giant component for time frame:  318
Giant component for time frame:  319
Giant component for time frame:  320
Giant component for time frame:  321
Giant component for time frame:  322
Giant component for time frame:  323
Giant component for time frame:  324
Giant component for time frame:  325
Giant component for time frame:  326
Giant component for time frame:  327
Giant component for time frame:  328
Giant component for time frame:  329
Giant component for time frame:  330
Giant component for time frame:  331
Giant component for time frame:  332
Giant component for time frame:  333
G

In [28]:
# Desikan atlas - EEG alpha band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_desikan)):
    
    for file in files:
        
        if 'conn_desi_cohi_time_eeg_alpha' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_alpha','dsk',os.path.join(subdir, file), threshold_values)
            s += 1


Threshold analysis for subject 1, with eeg_alpha and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj01-7T/conn_desi_cohi_time_eeg_alpha_subj1-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 11
Number of nodes of the second biggest component: 5
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 12
Number of nodes of the second biggest component: 6
___________________________________________________________________________
Proportion kept: 1.6
Number of edges of thresholded graph: 36
Average degree of thresholded graph:  1.0588235294117647
Number of nodes of the giant component: 12
Number of nodes of the second biggest component: 6
_______________________________

Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 3, with eeg_alpha and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj03-7T/conn_desi_cohi_time_eeg_alpha_subj3-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 10
Number of nodes of the second biggest component: 5
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thres

Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 5, with eeg_alpha and using dsk
/strombolihome/fribeiro/Dataset/source_reconstru

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 7, with eeg_alpha and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj07-7T/conn_desi_cohi_time_eeg_alpha_subj7-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 10
Number of nodes of the second biggest component: 3
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 13
Number of nodes of the s

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 9, with eeg_alpha and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj09-7T/conn_desi_cohi_time_eeg_alpha_subj9-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 6
Number of nodes of the second biggest component: 4
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 9
Number of nodes of the sec

In [5]:
# Desikan atlas - EEG alpha band

#componentAnalysis(1,'eeg_alpha','dsk', dir_eeg_desikan + 'subj01-7T/conn_desi_cohi_time_eeg_alpha_subj1-7T_.mat', [0.03], '2D')
#componentAnalysis(2,'eeg_alpha','dsk', dir_eeg_desikan + 'subj02-7T/conn_desi_cohi_time_eeg_alpha_subj2-7T_.mat', [0.03], '2D')
#componentAnalysis(3,'eeg_alpha','dsk', dir_eeg_desikan + 'subj03-7T/conn_desi_cohi_time_eeg_alpha_subj3-7T_.mat', [0.03], '2D')
#componentAnalysis(4,'eeg_alpha','dsk', dir_eeg_desikan + 'subj04-7T/conn_desi_cohi_time_eeg_alpha_subj4-7T_.mat', [0.03], '2D')
#componentAnalysis(5,'eeg_alpha','dsk', dir_eeg_desikan + 'subj05-7T/conn_desi_cohi_time_eeg_alpha_subj5-7T_.mat', [0.03], '2D')
#componentAnalysis(6,'eeg_alpha','dsk', dir_eeg_desikan + 'subj06-7T/conn_desi_cohi_time_eeg_alpha_subj6-7T_.mat', [0.03], '2D')
#componentAnalysis(7,'eeg_alpha','dsk', dir_eeg_desikan + 'subj07-7T/conn_desi_cohi_time_eeg_alpha_subj7-7T_.mat', [0.03], '2D')
componentAnalysis(8,'eeg_alpha','dsk', dir_eeg_desikan + 'subj08-7T/conn_desi_cohi_time_eeg_alpha_subj8-7T_.mat', [0.03], '2D')
componentAnalysis(9,'eeg_alpha','dsk', dir_eeg_desikan + 'subj09-7T/conn_desi_cohi_time_eeg_alpha_subj9-7T_.mat', [0.03], '2D')

#componentAnalysis(1,'eeg_alpha','dsk', dir_eeg_desikan + 'subj01-7T/conn_desi_cohi_time_eeg_alpha_subj1-7T_.mat', [0.03], '3D')
#componentAnalysis(2,'eeg_alpha','dsk', dir_eeg_desikan + 'subj02-7T/conn_desi_cohi_time_eeg_alpha_subj2-7T_.mat', [0.03], '3D')
#componentAnalysis(3,'eeg_alpha','dsk', dir_eeg_desikan + 'subj03-7T/conn_desi_cohi_time_eeg_alpha_subj3-7T_.mat', [0.03], '3D')
#componentAnalysis(4,'eeg_alpha','dsk', dir_eeg_desikan + 'subj04-7T/conn_desi_cohi_time_eeg_alpha_subj4-7T_.mat', [0.03], '3D')
#componentAnalysis(5,'eeg_alpha','dsk', dir_eeg_desikan + 'subj05-7T/conn_desi_cohi_time_eeg_alpha_subj5-7T_.mat', [0.03], '3D')
#componentAnalysis(6,'eeg_alpha','dsk', dir_eeg_desikan + 'subj06-7T/conn_desi_cohi_time_eeg_alpha_subj6-7T_.mat', [0.03], '3D')
#componentAnalysis(7,'eeg_alpha','dsk', dir_eeg_desikan + 'subj07-7T/conn_desi_cohi_time_eeg_alpha_subj7-7T_.mat', [0.03], '3D')
#componentAnalysis(8,'eeg_alpha','dsk', dir_eeg_desikan + 'subj08-7T/conn_desi_cohi_time_eeg_alpha_subj8-7T_.mat', [0.03], '3D')
#componentAnalysis(9,'eeg_alpha','dsk', dir_eeg_desikan + 'subj09-7T/conn_desi_cohi_time_eeg_alpha_subj9-7T_.mat', [0.03], '3D')

Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant component for time frame:  7
Giant component for time frame:  8
Giant component for time frame:  9
Giant component for time frame:  10
Giant component for time frame:  11
Giant component for time frame:  12
Giant component for time frame:  13
Giant component for time frame:  14
Giant component for time frame:  15
Giant component for time frame:  16
Giant component for time frame:  17
Giant component for time frame:  18
Giant component for time frame:  19
Giant component for time frame:  20
Giant component for time frame:  21
Giant component for time frame:  22
Giant component for time frame:  23
Giant component for time frame:  24
Giant component for time frame:  25
Giant component for time frame:  26
Giant component for time frame:  27
Giant component for time frame:  28
G

Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
Giant component for time frame:  232
Giant component for time frame:  233
Giant component for time frame:  234
Giant component for time frame:  235
Giant component for time frame:  236
Giant component for time frame:  237
Giant component for time frame:  238
Giant component for time frame:  239
Giant component for time frame:  240
Giant component for time frame:  241
Giant component for time frame:  242
Giant component for time frame:  243
Giant component for time frame:  244
Giant component for time frame:  245
Giant component for time frame:  246
Giant component for time frame:  247
Giant component for time frame:  248
Giant component for time frame:  249
Giant component for time frame:  250
Giant component for time frame:  251
Giant component for time frame:  252
G

Giant component for time frame:  85
Giant component for time frame:  86
Giant component for time frame:  87
Giant component for time frame:  88
Giant component for time frame:  89
Giant component for time frame:  90
Giant component for time frame:  91
Giant component for time frame:  92
Giant component for time frame:  93
Giant component for time frame:  94
Giant component for time frame:  95
Giant component for time frame:  96
Giant component for time frame:  97
Giant component for time frame:  98
Giant component for time frame:  99
Giant component for time frame:  100
Giant component for time frame:  101
Giant component for time frame:  102
Giant component for time frame:  103
Giant component for time frame:  104
Giant component for time frame:  105
Giant component for time frame:  106
Giant component for time frame:  107
Giant component for time frame:  108
Giant component for time frame:  109
Giant component for time frame:  110
Giant component for time frame:  111
Giant component 

Giant component for time frame:  307
Giant component for time frame:  308
Giant component for time frame:  309
Giant component for time frame:  310
Giant component for time frame:  311
Giant component for time frame:  312
Giant component for time frame:  313
Giant component for time frame:  314
Giant component for time frame:  315
Giant component for time frame:  316
Giant component for time frame:  317
Giant component for time frame:  318
Giant component for time frame:  319
Giant component for time frame:  320
Giant component for time frame:  321
Giant component for time frame:  322
Giant component for time frame:  323
Giant component for time frame:  324
Giant component for time frame:  325
Giant component for time frame:  326
Giant component for time frame:  327
Giant component for time frame:  328
Giant component for time frame:  329
Giant component for time frame:  330
Giant component for time frame:  331
Giant component for time frame:  332
Giant component for time frame:  333
G

In [29]:
# Desikan atlas - EEG beta band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_desikan)):
    
    for file in files:
        
        if 'conn_desi_cohi_time_eeg_beta' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_beta','dsk',os.path.join(subdir, file), threshold_values)
            s += 1


Threshold analysis for subject 1, with eeg_beta and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj01-7T/conn_desi_cohi_time_eeg_beta_subj1-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 7
Number of nodes of the second biggest component: 5
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 8
Number of nodes of the second biggest component: 7
___________________________________________________________________________
Proportion kept: 1.6
Number of edges of thresholded graph: 36
Average degree of thresholded graph:  1.0588235294117647
Number of nodes of the giant component: 12
Number of nodes of the second biggest component: 7
___________________________________

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 3, with eeg_beta and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj03-7T/conn_desi_cohi_time_eeg_beta_subj3-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 10
Number of nodes of the second biggest component: 3
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 13
Number of nodes of the sec

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 5, with eeg_beta and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj05-7T/conn_desi_cohi_time_eeg_beta_subj5-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 11
Number of nodes of the second biggest component: 5
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 21
Number of nodes of the sec

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 7, with eeg_beta and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj07-7T/conn_desi_cohi_time_eeg_beta_subj7-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 6
Number of nodes of the second biggest component: 3
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 9
Number of nodes of the secon

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 9, with eeg_beta and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj09-7T/conn_desi_cohi_time_eeg_beta_subj9-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 5
Number of nodes of the second biggest component: 5
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 18
Number of nodes of the seco

In [5]:
# Desikan atlas - EEG beta band

#componentAnalysis(1,'eeg_beta','dsk', dir_eeg_desikan + 'subj01-7T/conn_desi_cohi_time_eeg_beta_subj1-7T_.mat', [0.03], '2D')
#componentAnalysis(2,'eeg_beta','dsk', dir_eeg_desikan + 'subj02-7T/conn_desi_cohi_time_eeg_beta_subj2-7T_.mat', [0.03], '2D')
#componentAnalysis(3,'eeg_beta','dsk', dir_eeg_desikan + 'subj03-7T/conn_desi_cohi_time_eeg_beta_subj3-7T_.mat', [0.03], '2D')
#componentAnalysis(4,'eeg_beta','dsk', dir_eeg_desikan + 'subj04-7T/conn_desi_cohi_time_eeg_beta_subj4-7T_.mat', [0.03], '2D')
#componentAnalysis(5,'eeg_beta','dsk', dir_eeg_desikan + 'subj05-7T/conn_desi_cohi_time_eeg_beta_subj5-7T_.mat', [0.03], '2D')
#componentAnalysis(6,'eeg_beta','dsk', dir_eeg_desikan + 'subj06-7T/conn_desi_cohi_time_eeg_beta_subj6-7T_.mat', [0.03], '2D')
#componentAnalysis(7,'eeg_beta','dsk', dir_eeg_desikan + 'subj07-7T/conn_desi_cohi_time_eeg_beta_subj7-7T_.mat', [0.03], '2D')
componentAnalysis(8,'eeg_beta','dsk', dir_eeg_desikan + 'subj08-7T/conn_desi_cohi_time_eeg_beta_subj8-7T_.mat', [0.03], '2D')
componentAnalysis(9,'eeg_beta','dsk', dir_eeg_desikan + 'subj09-7T/conn_desi_cohi_time_eeg_beta_subj9-7T_.mat', [0.03], '2D')

#componentAnalysis(1,'eeg_beta','dsk', dir_eeg_desikan + 'subj01-7T/conn_desi_cohi_time_eeg_beta_subj1-7T_.mat', [0.03], '3D')
#componentAnalysis(2,'eeg_beta','dsk', dir_eeg_desikan + 'subj02-7T/conn_desi_cohi_time_eeg_beta_subj2-7T_.mat', [0.03], '3D')
#componentAnalysis(3,'eeg_beta','dsk', dir_eeg_desikan + 'subj03-7T/conn_desi_cohi_time_eeg_beta_subj3-7T_.mat', [0.03], '3D')
#componentAnalysis(4,'eeg_beta','dsk', dir_eeg_desikan + 'subj04-7T/conn_desi_cohi_time_eeg_beta_subj4-7T_.mat', [0.03], '3D')
#componentAnalysis(5,'eeg_beta','dsk', dir_eeg_desikan + 'subj05-7T/conn_desi_cohi_time_eeg_beta_subj5-7T_.mat', [0.03], '3D')
#componentAnalysis(6,'eeg_beta','dsk', dir_eeg_desikan + 'subj06-7T/conn_desi_cohi_time_eeg_beta_subj6-7T_.mat', [0.03], '3D')
#componentAnalysis(7,'eeg_beta','dsk', dir_eeg_desikan + 'subj07-7T/conn_desi_cohi_time_eeg_beta_subj7-7T_.mat', [0.03], '3D')
#componentAnalysis(8,'eeg_beta','dsk', dir_eeg_desikan + 'subj08-7T/conn_desi_cohi_time_eeg_beta_subj8-7T_.mat', [0.03], '3D')
#componentAnalysis(9,'eeg_beta','dsk', dir_eeg_desikan + 'subj09-7T/conn_desi_cohi_time_eeg_beta_subj9-7T_.mat', [0.03], '3D')

Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant component for time frame:  7
Giant component for time frame:  8
Giant component for time frame:  9
Giant component for time frame:  10
Giant component for time frame:  11
Giant component for time frame:  12
Giant component for time frame:  13
Giant component for time frame:  14
Giant component for time frame:  15
Giant component for time frame:  16
Giant component for time frame:  17
Giant component for time frame:  18
Giant component for time frame:  19
Giant component for time frame:  20
Giant component for time frame:  21
Giant component for time frame:  22
Giant component for time frame:  23
Giant component for time frame:  24
Giant component for time frame:  25
Giant component for time frame:  26
Giant component for time frame:  27
Giant component for time frame:  28
G

Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
Giant component for time frame:  232
Giant component for time frame:  233
Giant component for time frame:  234
Giant component for time frame:  235
Giant component for time frame:  236
Giant component for time frame:  237
Giant component for time frame:  238
Giant component for time frame:  239
Giant component for time frame:  240
Giant component for time frame:  241
Giant component for time frame:  242
Giant component for time frame:  243
Giant component for time frame:  244
Giant component for time frame:  245
Giant component for time frame:  246
Giant component for time frame:  247
Giant component for time frame:  248
Giant component for time frame:  249
Giant component for time frame:  250
Giant component for time frame:  251
Giant component for time frame:  252
G

Giant component for time frame:  85
Giant component for time frame:  86
Giant component for time frame:  87
Giant component for time frame:  88
Giant component for time frame:  89
Giant component for time frame:  90
Giant component for time frame:  91
Giant component for time frame:  92
Giant component for time frame:  93
Giant component for time frame:  94
Giant component for time frame:  95
Giant component for time frame:  96
Giant component for time frame:  97
Giant component for time frame:  98
Giant component for time frame:  99
Giant component for time frame:  100
Giant component for time frame:  101
Giant component for time frame:  102
Giant component for time frame:  103
Giant component for time frame:  104
Giant component for time frame:  105
Giant component for time frame:  106
Giant component for time frame:  107
Giant component for time frame:  108
Giant component for time frame:  109
Giant component for time frame:  110
Giant component for time frame:  111
Giant component 

Giant component for time frame:  307
Giant component for time frame:  308
Giant component for time frame:  309
Giant component for time frame:  310
Giant component for time frame:  311
Giant component for time frame:  312
Giant component for time frame:  313
Giant component for time frame:  314
Giant component for time frame:  315
Giant component for time frame:  316
Giant component for time frame:  317
Giant component for time frame:  318
Giant component for time frame:  319
Giant component for time frame:  320
Giant component for time frame:  321
Giant component for time frame:  322
Giant component for time frame:  323
Giant component for time frame:  324
Giant component for time frame:  325
Giant component for time frame:  326
Giant component for time frame:  327
Giant component for time frame:  328
Giant component for time frame:  329
Giant component for time frame:  330
Giant component for time frame:  331
Giant component for time frame:  332
Giant component for time frame:  333
G

In [ ]:
# Desikan atlas - EEG delta band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_desikan)):
    
    for file in files:
        
        if 'conn_desi_cohi_time_eeg_delta' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_delta','dsk',os.path.join(subdir, file), threshold_values)
            s += 1


In [5]:
# Desikan atlas - EEG delta band

#componentAnalysis(1,'eeg_delta','dsk', dir_eeg_desikan + 'subj01-7T/conn_desi_cohi_time_eeg_delta_subj1-7T_.mat', [0.03], '2D')
#componentAnalysis(2,'eeg_delta','dsk', dir_eeg_desikan + 'subj02-7T/conn_desi_cohi_time_eeg_delta_subj2-7T_.mat', [0.03], '2D')
componentAnalysis(3,'eeg_delta','dsk', dir_eeg_desikan + 'subj03-7T/conn_desi_cohi_time_eeg_delta_subj3-7T_.mat', [0.03], '2D')
componentAnalysis(4,'eeg_delta','dsk', dir_eeg_desikan + 'subj04-7T/conn_desi_cohi_time_eeg_delta_subj4-7T_.mat', [0.03], '2D')
componentAnalysis(5,'eeg_delta','dsk', dir_eeg_desikan + 'subj05-7T/conn_desi_cohi_time_eeg_delta_subj5-7T_.mat', [0.03], '2D')
#componentAnalysis(6,'eeg_delta','dsk', dir_eeg_desikan + 'subj06-7T/conn_desi_cohi_time_eeg_delta_subj6-7T_.mat', [0.03], '2D')
#componentAnalysis(7,'eeg_delta','dsk', dir_eeg_desikan + 'subj07-7T/conn_desi_cohi_time_eeg_delta_subj7-7T_.mat', [0.03], '2D')
#componentAnalysis(8,'eeg_delta','dsk', dir_eeg_desikan + 'subj08-7T/conn_desi_cohi_time_eeg_delta_subj8-7T_.mat', [0.03], '2D')
#componentAnalysis(9,'eeg_delta','dsk', dir_eeg_desikan + 'subj09-7T/conn_desi_cohi_time_eeg_delta_subj9-7T_.mat', [0.03], '2D')

#componentAnalysis(1,'eeg_delta','dsk', dir_eeg_desikan + 'subj01-7T/conn_desi_cohi_time_eeg_delta_subj1-7T_.mat', [0.03], '3D')
#componentAnalysis(2,'eeg_delta','dsk', dir_eeg_desikan + 'subj02-7T/conn_desi_cohi_time_eeg_delta_subj2-7T_.mat', [0.03], '3D')
#componentAnalysis(3,'eeg_delta','dsk', dir_eeg_desikan + 'subj03-7T/conn_desi_cohi_time_eeg_delta_subj3-7T_.mat', [0.03], '3D')
#componentAnalysis(4,'eeg_delta','dsk', dir_eeg_desikan + 'subj04-7T/conn_desi_cohi_time_eeg_delta_subj4-7T_.mat', [0.03], '3D')
#componentAnalysis(5,'eeg_delta','dsk', dir_eeg_desikan + 'subj05-7T/conn_desi_cohi_time_eeg_delta_subj5-7T_.mat', [0.03], '3D')
#componentAnalysis(6,'eeg_delta','dsk', dir_eeg_desikan + 'subj06-7T/conn_desi_cohi_time_eeg_delta_subj6-7T_.mat', [0.03], '3D')
#componentAnalysis(7,'eeg_delta','dsk', dir_eeg_desikan + 'subj07-7T/conn_desi_cohi_time_eeg_delta_subj7-7T_.mat', [0.03], '3D')
#componentAnalysis(8,'eeg_delta','dsk', dir_eeg_desikan + 'subj08-7T/conn_desi_cohi_time_eeg_delta_subj8-7T_.mat', [0.03], '3D')
#componentAnalysis(9,'eeg_delta','dsk', dir_eeg_desikan + 'subj09-7T/conn_desi_cohi_time_eeg_delta_subj9-7T_.mat', [0.03], '3D')

Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant component for time frame:  7
Giant component for time frame:  8
Giant component for time frame:  9
Giant component for time frame:  10
Giant component for time frame:  11
Giant component for time frame:  12
Giant component for time frame:  13
Giant component for time frame:  14
Giant component for time frame:  15
Giant component for time frame:  16
Giant component for time frame:  17
Giant component for time frame:  18
Giant component for time frame:  19
Giant component for time frame:  20
Giant component for time frame:  21
Giant component for time frame:  22
Giant component for time frame:  23
Giant component for time frame:  24
Giant component for time frame:  25
Giant component for time frame:  26
Giant component for time frame:  27
Giant component for time frame:  28
G

Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
Giant component for time frame:  232
Giant component for time frame:  233
Giant component for time frame:  234
Giant component for time frame:  235
Giant component for time frame:  236
Giant component for time frame:  237
Giant component for time frame:  238
Giant component for time frame:  239
Giant component for time frame:  240
Giant component for time frame:  241
Giant component for time frame:  242
Giant component for time frame:  243
Giant component for time frame:  244
Giant component for time frame:  245
Giant component for time frame:  246
Giant component for time frame:  247
Giant component for time frame:  248
Giant component for time frame:  249
Giant component for time frame:  250
Giant component for time frame:  251
Giant component for time frame:  252
G

Giant component for time frame:  49
Giant component for time frame:  50
Giant component for time frame:  51
Giant component for time frame:  52
Giant component for time frame:  53
Giant component for time frame:  54
Giant component for time frame:  55
Giant component for time frame:  56
Giant component for time frame:  57
Giant component for time frame:  58
Giant component for time frame:  59
Giant component for time frame:  60
Giant component for time frame:  61
Giant component for time frame:  62
Giant component for time frame:  63
Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time frame:  74
Giant component for time frame:  75
Giant component for time fra

Giant component for time frame:  272
Giant component for time frame:  273
Giant component for time frame:  274
Giant component for time frame:  275
Giant component for time frame:  276
Giant component for time frame:  277
Giant component for time frame:  278
Giant component for time frame:  279
Giant component for time frame:  280
Giant component for time frame:  281
Giant component for time frame:  282
Giant component for time frame:  283
Giant component for time frame:  284
Giant component for time frame:  285
Giant component for time frame:  286
Giant component for time frame:  287
Giant component for time frame:  288
Giant component for time frame:  289
Giant component for time frame:  290
Giant component for time frame:  291
Giant component for time frame:  292
Giant component for time frame:  293
Giant component for time frame:  294
Giant component for time frame:  295
Giant component for time frame:  296
Giant component for time frame:  297
Giant component for time frame:  298
G

In [3]:
# Desikan atlas - EEG gamma band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_desikan)):
    
    for file in files:
        
        if 'conn_desi_cohi_time_eeg_gamma' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_gamma','dsk',os.path.join(subdir, file), threshold_values)
            s += 1


NameError: name 'thresholdAnalysis' is not defined

In [12]:
# Desikan atlas - EEG gamma band

#componentAnalysis(1,'eeg_gamma','dsk', dir_eeg_desikan + 'subj01-7T/conn_desi_cohi_time_eeg_gamma_subj1-7T_.mat', [0.03], '2D')
#componentAnalysis(2,'eeg_gamma','dsk', dir_eeg_desikan + 'subj02-7T/conn_desi_cohi_time_eeg_gamma_subj2-7T_.mat', [0.03], '2D')
#componentAnalysis(3,'eeg_gamma','dsk', dir_eeg_desikan + 'subj03-7T/conn_desi_cohi_time_eeg_gamma_subj3-7T_.mat', [0.03], '2D')
#componentAnalysis(4,'eeg_gamma','dsk', dir_eeg_desikan + 'subj04-7T/conn_desi_cohi_time_eeg_gamma_subj4-7T_.mat', [0.03], '2D')
#componentAnalysis(5,'eeg_gamma','dsk', dir_eeg_desikan + 'subj05-7T/conn_desi_cohi_time_eeg_gamma_subj5-7T_.mat', [0.03], '2D')
#componentAnalysis(6,'eeg_gamma','dsk', dir_eeg_desikan + 'subj06-7T/conn_desi_cohi_time_eeg_gamma_subj6-7T_.mat', [0.03], '2D')
#componentAnalysis(7,'eeg_gamma','dsk', dir_eeg_desikan + 'subj07-7T/conn_desi_cohi_time_eeg_gamma_subj7-7T_.mat', [0.03], '2D')
#componentAnalysis(8,'eeg_gamma','dsk', dir_eeg_desikan + 'subj08-7T/conn_desi_cohi_time_eeg_gamma_subj8-7T_.mat', [0.03], '2D')
#componentAnalysis(9,'eeg_gamma','dsk', dir_eeg_desikan + 'subj09-7T/conn_desi_cohi_time_eeg_gamma_subj9-7T_.mat', [0.03], '2D')

componentAnalysis(1,'eeg_gamma','dsk', dir_eeg_desikan + 'subj01-7T/conn_desi_cohi_time_eeg_gamma_subj1-7T_.mat', [0.03], '3D')
componentAnalysis(2,'eeg_gamma','dsk', dir_eeg_desikan + 'subj02-7T/conn_desi_cohi_time_eeg_gamma_subj2-7T_.mat', [0.03], '3D')
componentAnalysis(3,'eeg_gamma','dsk', dir_eeg_desikan + 'subj03-7T/conn_desi_cohi_time_eeg_gamma_subj3-7T_.mat', [0.03], '3D')
componentAnalysis(4,'eeg_gamma','dsk', dir_eeg_desikan + 'subj04-7T/conn_desi_cohi_time_eeg_gamma_subj4-7T_.mat', [0.03], '3D')
componentAnalysis(5,'eeg_gamma','dsk', dir_eeg_desikan + 'subj05-7T/conn_desi_cohi_time_eeg_gamma_subj5-7T_.mat', [0.03], '3D')
componentAnalysis(6,'eeg_gamma','dsk', dir_eeg_desikan + 'subj06-7T/conn_desi_cohi_time_eeg_gamma_subj6-7T_.mat', [0.03], '3D')
componentAnalysis(7,'eeg_gamma','dsk', dir_eeg_desikan + 'subj07-7T/conn_desi_cohi_time_eeg_gamma_subj7-7T_.mat', [0.03], '3D')
componentAnalysis(8,'eeg_gamma','dsk', dir_eeg_desikan + 'subj08-7T/conn_desi_cohi_time_eeg_gamma_subj8-7T_.mat', [0.03], '3D')
componentAnalysis(9,'eeg_gamma','dsk', dir_eeg_desikan + 'subj09-7T/conn_desi_cohi_time_eeg_gamma_subj9-7T_.mat', [0.03], '3D')


Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant component for time frame:  7
Giant component for time frame:  8
Giant component for time frame:  9
Giant component for time frame:  10
Giant component for time frame:  11
Giant component for time frame:  12
Giant component for time frame:  13
Giant component for time frame:  14
Giant component for time frame:  15
Giant component for time frame:  16
Giant component for time frame:  17
Giant component for time frame:  18
Giant component for time frame:  19
Giant component for time frame:  20
Giant component for time frame:  21
Giant component for time frame:  22
Giant component for time frame:  23
Giant component for time frame:  24
Giant component for time frame:  25
Giant component for time frame:  26
Giant component for time frame:  27
Giant component for time frame:  28
G

Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
Giant component for time frame:  232
Giant component for time frame:  233
Giant component for time frame:  234
Giant component for time frame:  235
Giant component for time frame:  236
Giant component for time frame:  237
Giant component for time frame:  238
Giant component for time frame:  239
Giant component for time frame:  240
Giant component for time frame:  241
Giant component for time frame:  242
Giant component for time frame:  243
Giant component for time frame:  244
Giant component for time frame:  245
Giant component for time frame:  246
Giant component for time frame:  247
Giant component for time frame:  248
Giant component for time frame:  249
Giant component for time frame:  250
Giant component for time frame:  251
Giant component for time frame:  252
G

Giant component for time frame:  49
Giant component for time frame:  50
Giant component for time frame:  51
Giant component for time frame:  52
Giant component for time frame:  53
Giant component for time frame:  54
Giant component for time frame:  55
Giant component for time frame:  56
Giant component for time frame:  57
Giant component for time frame:  58
Giant component for time frame:  59
Giant component for time frame:  60
Giant component for time frame:  61
Giant component for time frame:  62
Giant component for time frame:  63
Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time frame:  74
Giant component for time frame:  75
Giant component for time fra

Giant component for time frame:  272
Giant component for time frame:  273
Giant component for time frame:  274
Giant component for time frame:  275
Giant component for time frame:  276
Giant component for time frame:  277
Giant component for time frame:  278
Giant component for time frame:  279
Giant component for time frame:  280
Giant component for time frame:  281
Giant component for time frame:  282
Giant component for time frame:  283
Giant component for time frame:  284
Giant component for time frame:  285
Giant component for time frame:  286
Giant component for time frame:  287
Giant component for time frame:  288
Giant component for time frame:  289
Giant component for time frame:  290
Giant component for time frame:  291
Giant component for time frame:  292
Giant component for time frame:  293
Giant component for time frame:  294
Giant component for time frame:  295
Giant component for time frame:  296
Giant component for time frame:  297
Giant component for time frame:  298
G

Giant component for time frame:  47
Giant component for time frame:  48
Giant component for time frame:  49
Giant component for time frame:  50
Giant component for time frame:  51
Giant component for time frame:  52
Giant component for time frame:  53
Giant component for time frame:  54
Giant component for time frame:  55
Giant component for time frame:  56
Giant component for time frame:  57
Giant component for time frame:  58
Giant component for time frame:  59
Giant component for time frame:  60
Giant component for time frame:  61
Giant component for time frame:  62
Giant component for time frame:  63
Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time fra

...

Giant component for time frame:  165
Giant component for time frame:  166
Giant component for time frame:  167
Giant component for time frame:  168
Giant component for time frame:  169
Giant component for time frame:  170
Giant component for time frame:  171
Giant component for time frame:  172
Giant component for time frame:  173
Giant component for time frame:  174
Giant component for time frame:  175
Giant component for time frame:  176
Giant component for time frame:  177
Giant component for time frame:  178
Giant component for time frame:  179
Giant component for time frame:  180
Giant component for time frame:  181
Giant component for time frame:  182
Giant component for time frame:  183
Giant component for time frame:  184
Giant component for time frame:  185
Giant component for time frame:  186
Giant component for time frame:  187
Giant component for time frame:  188
Giant component for time frame:  189
Giant component for time frame:  190
Giant component for time frame:  191
G

Giant component for time frame:  387
Giant component for time frame:  388
Giant component for time frame:  389
Giant component for time frame:  390
Giant component for time frame:  391
Giant component for time frame:  392
Giant component for time frame:  393
Giant component for time frame:  394
Giant component for time frame:  395
Giant component for time frame:  396
Giant component for time frame:  397
Giant component for time frame:  398
Giant component for time frame:  399
Giant component for time frame:  400
Giant component for time frame:  401
Giant component for time frame:  402
Giant component for time frame:  403
Giant component for time frame:  404
Giant component for time frame:  405
Giant component for time frame:  406
Giant component for time frame:  407
Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant compone

Giant component for time frame:  205
Giant component for time frame:  206
Giant component for time frame:  207
Giant component for time frame:  208
Giant component for time frame:  209
Giant component for time frame:  210
Giant component for time frame:  211
Giant component for time frame:  212
Giant component for time frame:  213
Giant component for time frame:  214
Giant component for time frame:  215
Giant component for time frame:  216
Giant component for time frame:  217
Giant component for time frame:  218
Giant component for time frame:  219
Giant component for time frame:  220
Giant component for time frame:  221
Giant component for time frame:  222
Giant component for time frame:  223
Giant component for time frame:  224
Giant component for time frame:  225
Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
G

Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time frame:  74
Giant component for time frame:  75
Giant component for time frame:  76
Giant component for time frame:  77
Giant component for time frame:  78
Giant component for time frame:  79
Giant component for time frame:  80
Giant component for time frame:  81
Giant component for time frame:  82
Giant component for time frame:  83
Giant component for time frame:  84
Giant component for time frame:  85
Giant component for time frame:  86
Giant component for time frame:  87
Giant component for time frame:  88
Giant component for time frame:  89
Giant component for time frame:  90
Giant component for time fra

Giant component for time frame:  287
Giant component for time frame:  288
Giant component for time frame:  289
Giant component for time frame:  290
Giant component for time frame:  291
Giant component for time frame:  292
Giant component for time frame:  293
Giant component for time frame:  294
Giant component for time frame:  295
Giant component for time frame:  296
Giant component for time frame:  297
Giant component for time frame:  298
Giant component for time frame:  299
Giant component for time frame:  300
Giant component for time frame:  301
Giant component for time frame:  302
Giant component for time frame:  303
Giant component for time frame:  304
Giant component for time frame:  305
Giant component for time frame:  306
Giant component for time frame:  307
Giant component for time frame:  308
Giant component for time frame:  309
Giant component for time frame:  310
Giant component for time frame:  311
Giant component for time frame:  312
Giant component for time frame:  313
G

In [ ]:
# Desikan atlas - EEG theta band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_desikan)):
    
    for file in files:
        
        if 'conn_desi_cohi_time_eeg_theta' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_theta','dsk',os.path.join(subdir, file), threshold_values)
            s += 1


In [13]:
# Desikan atlas - EEG theta band

#componentAnalysis(1,'eeg_theta','dsk', dir_eeg_desikan + 'subj01-7T/conn_desi_cohi_time_eeg_theta_subj1-7T_.mat', [0.03], '2D')
#componentAnalysis(2,'eeg_theta','dsk', dir_eeg_desikan + 'subj02-7T/conn_desi_cohi_time_eeg_theta_subj2-7T_.mat', [0.03], '2D')
#componentAnalysis(3,'eeg_theta','dsk', dir_eeg_desikan + 'subj03-7T/conn_desi_cohi_time_eeg_theta_subj3-7T_.mat', [0.03], '2D')
#componentAnalysis(4,'eeg_theta','dsk', dir_eeg_desikan + 'subj04-7T/conn_desi_cohi_time_eeg_theta_subj4-7T_.mat', [0.03], '2D')
#componentAnalysis(5,'eeg_theta','dsk', dir_eeg_desikan + 'subj05-7T/conn_desi_cohi_time_eeg_theta_subj5-7T_.mat', [0.03], '2D')
#componentAnalysis(6,'eeg_theta','dsk', dir_eeg_desikan + 'subj06-7T/conn_desi_cohi_time_eeg_theta_subj6-7T_.mat', [0.03], '2D')
#componentAnalysis(7,'eeg_theta','dsk', dir_eeg_desikan + 'subj07-7T/conn_desi_cohi_time_eeg_theta_subj7-7T_.mat', [0.03], '2D')
#componentAnalysis(8,'eeg_theta','dsk', dir_eeg_desikan + 'subj08-7T/conn_desi_cohi_time_eeg_theta_subj8-7T_.mat', [0.03], '2D')
#componentAnalysis(9,'eeg_theta','dsk', dir_eeg_desikan + 'subj09-7T/conn_desi_cohi_time_eeg_theta_subj9-7T_.mat', [0.03], '2D')

componentAnalysis(1,'eeg_theta','dsk', dir_eeg_desikan + 'subj01-7T/conn_desi_cohi_time_eeg_theta_subj1-7T_.mat', [0.03], '3D')
componentAnalysis(2,'eeg_theta','dsk', dir_eeg_desikan + 'subj02-7T/conn_desi_cohi_time_eeg_theta_subj2-7T_.mat', [0.03], '3D')
componentAnalysis(3,'eeg_theta','dsk', dir_eeg_desikan + 'subj03-7T/conn_desi_cohi_time_eeg_theta_subj3-7T_.mat', [0.03], '3D')
componentAnalysis(4,'eeg_theta','dsk', dir_eeg_desikan + 'subj04-7T/conn_desi_cohi_time_eeg_theta_subj4-7T_.mat', [0.03], '3D')
componentAnalysis(5,'eeg_theta','dsk', dir_eeg_desikan + 'subj05-7T/conn_desi_cohi_time_eeg_theta_subj5-7T_.mat', [0.03], '3D')
componentAnalysis(6,'eeg_theta','dsk', dir_eeg_desikan + 'subj06-7T/conn_desi_cohi_time_eeg_theta_subj6-7T_.mat', [0.03], '3D')
componentAnalysis(7,'eeg_theta','dsk', dir_eeg_desikan + 'subj07-7T/conn_desi_cohi_time_eeg_theta_subj7-7T_.mat', [0.03], '3D')
componentAnalysis(8,'eeg_theta','dsk', dir_eeg_desikan + 'subj08-7T/conn_desi_cohi_time_eeg_theta_subj8-7T_.mat', [0.03], '3D')
componentAnalysis(9,'eeg_theta','dsk', dir_eeg_desikan + 'subj09-7T/conn_desi_cohi_time_eeg_theta_subj9-7T_.mat', [0.03], '3D')

Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant component for time frame:  7
Giant component for time frame:  8
Giant component for time frame:  9
Giant component for time frame:  10
Giant component for time frame:  11
Giant component for time frame:  12
Giant component for time frame:  13
Giant component for time frame:  14
Giant component for time frame:  15
Giant component for time frame:  16
Giant component for time frame:  17
Giant component for time frame:  18
Giant component for time frame:  19
Giant component for time frame:  20
Giant component for time frame:  21
Giant component for time frame:  22
Giant component for time frame:  23
Giant component for time frame:  24
Giant component for time frame:  25
Giant component for time frame:  26
Giant component for time frame:  27
Giant component for time frame:  28
G

Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
Giant component for time frame:  232
Giant component for time frame:  233
Giant component for time frame:  234
Giant component for time frame:  235
Giant component for time frame:  236
Giant component for time frame:  237
Giant component for time frame:  238
Giant component for time frame:  239
Giant component for time frame:  240
Giant component for time frame:  241
Giant component for time frame:  242
Giant component for time frame:  243
Giant component for time frame:  244
Giant component for time frame:  245
Giant component for time frame:  246
Giant component for time frame:  247
Giant component for time frame:  248
Giant component for time frame:  249
Giant component for time frame:  250
Giant component for time frame:  251
Giant component for time frame:  252
G

Giant component for time frame:  49
Giant component for time frame:  50
Giant component for time frame:  51
Giant component for time frame:  52
Giant component for time frame:  53
Giant component for time frame:  54
Giant component for time frame:  55
Giant component for time frame:  56
Giant component for time frame:  57
Giant component for time frame:  58
Giant component for time frame:  59
Giant component for time frame:  60
Giant component for time frame:  61
Giant component for time frame:  62
Giant component for time frame:  63
Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time frame:  74
Giant component for time frame:  75
Giant component for time fra

Giant component for time frame:  272
Giant component for time frame:  273
Giant component for time frame:  274
Giant component for time frame:  275
Giant component for time frame:  276
Giant component for time frame:  277
Giant component for time frame:  278
Giant component for time frame:  279
Giant component for time frame:  280
Giant component for time frame:  281
Giant component for time frame:  282
Giant component for time frame:  283
Giant component for time frame:  284
Giant component for time frame:  285
Giant component for time frame:  286
Giant component for time frame:  287
Giant component for time frame:  288
Giant component for time frame:  289
Giant component for time frame:  290
Giant component for time frame:  291
Giant component for time frame:  292
Giant component for time frame:  293
Giant component for time frame:  294
Giant component for time frame:  295
Giant component for time frame:  296
Giant component for time frame:  297
Giant component for time frame:  298
G

Giant component for time frame:  47
Giant component for time frame:  48
Giant component for time frame:  49
Giant component for time frame:  50
Giant component for time frame:  51
Giant component for time frame:  52
Giant component for time frame:  53
Giant component for time frame:  54
Giant component for time frame:  55
Giant component for time frame:  56
Giant component for time frame:  57
Giant component for time frame:  58
Giant component for time frame:  59
Giant component for time frame:  60
Giant component for time frame:  61
Giant component for time frame:  62
Giant component for time frame:  63
Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time fra

...

Giant component for time frame:  165
Giant component for time frame:  166
Giant component for time frame:  167
Giant component for time frame:  168
Giant component for time frame:  169
Giant component for time frame:  170
Giant component for time frame:  171
Giant component for time frame:  172
Giant component for time frame:  173
Giant component for time frame:  174
Giant component for time frame:  175
Giant component for time frame:  176
Giant component for time frame:  177
Giant component for time frame:  178
Giant component for time frame:  179
Giant component for time frame:  180
Giant component for time frame:  181
Giant component for time frame:  182
Giant component for time frame:  183
Giant component for time frame:  184
Giant component for time frame:  185
Giant component for time frame:  186
Giant component for time frame:  187
Giant component for time frame:  188
Giant component for time frame:  189
Giant component for time frame:  190
Giant component for time frame:  191
G

Giant component for time frame:  387
Giant component for time frame:  388
Giant component for time frame:  389
Giant component for time frame:  390
Giant component for time frame:  391
Giant component for time frame:  392
Giant component for time frame:  393
Giant component for time frame:  394
Giant component for time frame:  395
Giant component for time frame:  396
Giant component for time frame:  397
Giant component for time frame:  398
Giant component for time frame:  399
Giant component for time frame:  400
Giant component for time frame:  401
Giant component for time frame:  402
Giant component for time frame:  403
Giant component for time frame:  404
Giant component for time frame:  405
Giant component for time frame:  406
Giant component for time frame:  407
Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant compone

Giant component for time frame:  205
Giant component for time frame:  206
Giant component for time frame:  207
Giant component for time frame:  208
Giant component for time frame:  209
Giant component for time frame:  210
Giant component for time frame:  211
Giant component for time frame:  212
Giant component for time frame:  213
Giant component for time frame:  214
Giant component for time frame:  215
Giant component for time frame:  216
Giant component for time frame:  217
Giant component for time frame:  218
Giant component for time frame:  219
Giant component for time frame:  220
Giant component for time frame:  221
Giant component for time frame:  222
Giant component for time frame:  223
Giant component for time frame:  224
Giant component for time frame:  225
Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
G

Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time frame:  74
Giant component for time frame:  75
Giant component for time frame:  76
Giant component for time frame:  77
Giant component for time frame:  78
Giant component for time frame:  79
Giant component for time frame:  80
Giant component for time frame:  81
Giant component for time frame:  82
Giant component for time frame:  83
Giant component for time frame:  84
Giant component for time frame:  85
Giant component for time frame:  86
Giant component for time frame:  87
Giant component for time frame:  88
Giant component for time frame:  89
Giant component for time frame:  90
Giant component for time fra

Giant component for time frame:  287
Giant component for time frame:  288
Giant component for time frame:  289
Giant component for time frame:  290
Giant component for time frame:  291
Giant component for time frame:  292
Giant component for time frame:  293
Giant component for time frame:  294
Giant component for time frame:  295
Giant component for time frame:  296
Giant component for time frame:  297
Giant component for time frame:  298
Giant component for time frame:  299
Giant component for time frame:  300
Giant component for time frame:  301
Giant component for time frame:  302
Giant component for time frame:  303
Giant component for time frame:  304
Giant component for time frame:  305
Giant component for time frame:  306
Giant component for time frame:  307
Giant component for time frame:  308
Giant component for time frame:  309
Giant component for time frame:  310
Giant component for time frame:  311
Giant component for time frame:  312
Giant component for time frame:  313
G

In [36]:
# Destrieux atlas - EEG broad band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_destrieux)):
    
    for file in files:
        
        if 'conn_destr_cohi_time_eeg_broad' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg','dstrx',os.path.join(subdir, file), threshold_values)
            s += 1


conn_destr_cohi_time_eeg_theta_subj1-7T_.mat
conn_destr_cohi_time_eeg_alpha_subj1-7T_.mat
conn_destr_cohi_time_eeg_delta_subj1-7T_.mat
conn_destr_cohi_time_eeg_gamma_subj1-7T_.mat
conn_destr_cohi_time_eeg_beta_subj1-7T_.mat
conn_destr_cohi_time_eeg_broad_subj1-7T_.mat
Threshold analysis for subject 1, with eeg and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_destrieux/subj01-7T/conn_destr_cohi_time_eeg_broad_subj1-7T_.mat
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 8
Number of nodes of the second biggest component: 5
___________________________________________________________________________
Proportion kept: 0.8
Number of edges of thresholded graph: 87
Average degree of thresholded graph:  1.1756756756756757
Number of nodes of the giant component: 17
Number of nodes of the second biggest component: 15
_____________________________________

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
conn_destr_cohi_time_eeg_gamma_subj2-7T_.mat
conn_destr_cohi_time_eeg_beta_subj2-7T_.mat
conn_destr_cohi_time_eeg_alpha_subj2-7T_.mat
conn_destr_cohi_time_eeg_delta_subj3-7T_.mat
conn_destr_cohi_time_eeg_gamma_subj3-7T_.mat
conn_destr_cohi_time_eeg_theta_subj3-7T_.mat
conn_destr_cohi_time_eeg_alpha_subj3-7T_.mat
conn_des

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
conn_destr_cohi_time_eeg_delta_subj4-7T_.mat
conn_destr_cohi_time_eeg_theta_subj5-7T_.mat
conn_destr_cohi_time_eeg_broad_subj5-7T_.mat
Threshold analysis for subject 5, with eeg and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_destrieux/subj05-7T/conn_destr_cohi_time_eeg_broad_subj5-7T_

Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
conn_destr_cohi_time_eeg_alpha_subj6-7T_.mat
conn_destr_cohi_time_eeg_gamma_subj7-7T_.mat
conn_destr_cohi_time_eeg_beta_subj7-7T_.mat
conn_destr_cohi_time_eeg_broad_subj7-7T_.mat
Threshold analysis for subject 7, with eeg and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_destrieux/subj07-7T/conn_destr_cohi_time_eeg_broad_subj7-7T_.mat
Proportion kept: 0.5
Number of edges of thresholde

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
conn_destr_cohi_time_eeg_delta_subj8-7T_.mat
conn_destr_cohi_time_eeg_alpha_subj8-7T_.mat
conn_destr_cohi_time_eeg_beta_subj8-7T_.mat
conn_destr_cohi_time_eeg_gamma_subj8-7T_.mat
conn_destr_cohi_time_eeg_theta_subj8-7T_.mat
conn_destr_cohi_time_eeg_theta_subj9-7T_.mat
conn_destr_cohi_time_eeg_beta_subj9-7T_.mat
conn_dest

In [6]:
# Destrieux atlas - EEG broad band

#componentAnalysis(1,'eeg','dstrx', dir_eeg_destrieux + 'subj01-7T/conn_destr_cohi_time_eeg_broad_subj1-7T_.mat', [0.011], '2D')
componentAnalysis(2,'eeg','dstrx', dir_eeg_destrieux + 'subj02-7T/conn_destr_cohi_time_eeg_broad_subj2-7T_.mat', [0.011], '2D')
#componentAnalysis(3,'eeg','dstrx', dir_eeg_destrieux + 'subj03-7T/conn_destr_cohi_time_eeg_broad_subj3-7T_.mat', [0.011], '2D')
#componentAnalysis(4,'eeg','dstrx', dir_eeg_destrieux + 'subj04-7T/conn_destr_cohi_time_eeg_broad_subj4-7T_.mat', [0.011], '2D')
#componentAnalysis(5,'eeg','dstrx', dir_eeg_destrieux + 'subj05-7T/conn_destr_cohi_time_eeg_broad_subj5-7T_.mat', [0.011], '2D')
#componentAnalysis(6,'eeg','dstrx', dir_eeg_destrieux + 'subj06-7T/conn_destr_cohi_time_eeg_broad_subj6-7T_.mat', [0.011], '2D')
#componentAnalysis(7,'eeg','dstrx', dir_eeg_destrieux + 'subj07-7T/conn_destr_cohi_time_eeg_broad_subj7-7T_.mat', [0.011], '2D')
#componentAnalysis(8,'eeg','dstrx', dir_eeg_destrieux + 'subj08-7T/conn_destr_cohi_time_eeg_broad_subj8-7T_.mat', [0.011], '2D')
#componentAnalysis(9,'eeg','dstrx', dir_eeg_destrieux + 'subj09-7T/conn_destr_cohi_time_eeg_broad_subj9-7T_.mat', [0.011], '2D')

#componentAnalysis(1,'eeg','dstrx', dir_eeg_destrieux + 'subj01-7T/conn_destr_cohi_time_eeg_broad_subj1-7T_.mat', [0.011], '3D')
#componentAnalysis(2,'eeg','dstrx', dir_eeg_destrieux + 'subj02-7T/conn_destr_cohi_time_eeg_broad_subj2-7T_.mat', [0.011], '3D')
#componentAnalysis(3,'eeg','dstrx', dir_eeg_destrieux + 'subj03-7T/conn_destr_cohi_time_eeg_broad_subj3-7T_.mat', [0.011], '3D')
#componentAnalysis(4,'eeg','dstrx', dir_eeg_destrieux + 'subj04-7T/conn_destr_cohi_time_eeg_broad_subj4-7T_.mat', [0.011], '3D')
#componentAnalysis(5,'eeg','dstrx', dir_eeg_destrieux + 'subj05-7T/conn_destr_cohi_time_eeg_broad_subj5-7T_.mat', [0.011], '3D')
#componentAnalysis(6,'eeg','dstrx', dir_eeg_destrieux + 'subj06-7T/conn_destr_cohi_time_eeg_broad_subj6-7T_.mat', [0.011], '3D')
#componentAnalysis(7,'eeg','dstrx', dir_eeg_destrieux + 'subj07-7T/conn_destr_cohi_time_eeg_broad_subj7-7T_.mat', [0.011], '3D')
#componentAnalysis(8,'eeg','dstrx', dir_eeg_destrieux + 'subj08-7T/conn_destr_cohi_time_eeg_broad_subj8-7T_.mat', [0.011], '3D')
#componentAnalysis(9,'eeg','dstrx', dir_eeg_destrieux + 'subj09-7T/conn_destr_cohi_time_eeg_broad_subj9-7T_.mat', [0.011], '3D')


Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant component for time frame:  7
Giant component for time frame:  8
Giant component for time frame:  9
Giant component for time frame:  10
Giant component for time frame:  11
Giant component for time frame:  12
Giant component for time frame:  13
Giant component for time frame:  14
Giant component for time frame:  15
Giant component for time frame:  16
Giant component for time frame:  17
Giant component for time frame:  18
Giant component for time frame:  19
Giant component for time frame:  20
Giant component for time frame:  21
Giant component for time frame:  22
Giant component for time frame:  23
Giant component for time frame:  24
Giant component for time frame:  25
Giant component for time frame:  26
Giant component for time frame:  27
Giant component for time frame:  28
G

Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
Giant component for time frame:  232
Giant component for time frame:  233
Giant component for time frame:  234
Giant component for time frame:  235
Giant component for time frame:  236
Giant component for time frame:  237
Giant component for time frame:  238
Giant component for time frame:  239
Giant component for time frame:  240
Giant component for time frame:  241
Giant component for time frame:  242
Giant component for time frame:  243
Giant component for time frame:  244
Giant component for time frame:  245
Giant component for time frame:  246
Giant component for time frame:  247
Giant component for time frame:  248
Giant component for time frame:  249
Giant component for time frame:  250
Giant component for time frame:  251
Giant component for time frame:  252
G

In [37]:
# Destrieux atlas - EEG alpha band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_destrieux)):
    
    for file in files:
        
        if 'conn_destr_cohi_time_eeg_alpha' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_alpha','dstrx',os.path.join(subdir, file), threshold_values)
            s += 1


conn_destr_cohi_time_eeg_theta_subj1-7T_.mat
conn_destr_cohi_time_eeg_alpha_subj1-7T_.mat
Threshold analysis for subject 1, with eeg_alpha and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_destrieux/subj01-7T/conn_destr_cohi_time_eeg_alpha_subj1-7T_.mat
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 17
Number of nodes of the second biggest component: 10
___________________________________________________________________________
Proportion kept: 0.8
Number of edges of thresholded graph: 87
Average degree of thresholded graph:  1.1756756756756757
Number of nodes of the giant component: 28
Number of nodes of the second biggest component: 20
___________________________________________________________________________
Proportion kept: 1.0
Number of edges of thresholded graph: 109
Average degree of thresholded graph:  1.472972972972973
Number of nod

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
conn_destr_cohi_time_eeg_delta_subj3-7T_.mat
conn_destr_cohi_time_eeg_gamma_subj3-7T_.mat
conn_destr_cohi_time_eeg_theta_subj3-7T_.mat
conn_destr_cohi_time_eeg_alpha_subj3-7T_.mat
Threshold analysis for subject 3, with eeg_alpha and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_destrieux

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
conn_destr_cohi_time_eeg_broad_subj4-7T_.mat
conn_destr_cohi_time_eeg_delta_subj4-7T_.mat
conn_destr_cohi_time_eeg_theta_subj5-7T_.mat
conn_destr_cohi_time_eeg_broad_subj5-7T_.mat
conn_destr_cohi_time_eeg_delta_subj5-7T_.mat
conn_destr_cohi_time_eeg_alpha_subj5-7T_.mat
Threshold analysis for subject 5, with eeg_alpha and

Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
conn_destr_cohi_time_eeg_gamma_subj7-7T_.mat
conn_destr_cohi_time_eeg_beta_subj7-7T_.mat
conn_destr_cohi_time_eeg_broad_subj7-7T_.mat
conn_destr_cohi_time_eeg_delta_subj7-7T_.mat
conn_destr_cohi_time_eeg_theta_subj7-7T_.mat
conn_destr_cohi_time_eeg_alpha_subj7-7T_.mat
Threshold analysis for subject 7, with eeg_alpha and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_destrieux/subj07-7T

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
conn_destr_cohi_time_eeg_beta_subj8-7T_.mat
conn_destr_cohi_time_eeg_gamma_subj8-7T_.mat
conn_destr_cohi_time_eeg_theta_subj8-7T_.mat
conn_destr_cohi_time_eeg_theta_subj9-7T_.mat
conn_destr_cohi_time_eeg_beta_subj9-7T_.mat
conn_destr_cohi_time_eeg_delta_subj9-7T_.mat
conn_destr_cohi_time_eeg_gamma_subj9-7T_.mat
conn_dest

In [15]:
# Destrieux atlas - EEG alpha band

#componentAnalysis(1,'eeg_alpha','dstrx', dir_eeg_destrieux + 'subj01-7T/conn_destr_cohi_time_eeg_alpha_subj1-7T_.mat', [0.012], '2D')
#componentAnalysis(2,'eeg_alpha','dstrx', dir_eeg_destrieux + 'subj02-7T/conn_destr_cohi_time_eeg_alpha_subj2-7T_.mat', [0.012], '2D')
#componentAnalysis(3,'eeg_alpha','dstrx', dir_eeg_destrieux + 'subj03-7T/conn_destr_cohi_time_eeg_alpha_subj3-7T_.mat', [0.012], '2D')
#componentAnalysis(4,'eeg_alpha','dstrx', dir_eeg_destrieux + 'subj04-7T/conn_destr_cohi_time_eeg_alpha_subj4-7T_.mat', [0.012], '2D')
#componentAnalysis(5,'eeg_alpha','dstrx', dir_eeg_destrieux + 'subj05-7T/conn_destr_cohi_time_eeg_alpha_subj5-7T_.mat', [0.012], '2D')
#componentAnalysis(6,'eeg_alpha','dstrx', dir_eeg_destrieux + 'subj06-7T/conn_destr_cohi_time_eeg_alpha_subj6-7T_.mat', [0.012], '2D')
#componentAnalysis(7,'eeg_alpha','dstrx', dir_eeg_destrieux + 'subj07-7T/conn_destr_cohi_time_eeg_alpha_subj7-7T_.mat', [0.012], '2D')
#componentAnalysis(8,'eeg_alpha','dstrx', dir_eeg_destrieux + 'subj08-7T/conn_destr_cohi_time_eeg_alpha_subj8-7T_.mat', [0.012], '2D')
#componentAnalysis(9,'eeg_alpha','dstrx', dir_eeg_destrieux + 'subj09-7T/conn_destr_cohi_time_eeg_alpha_subj9-7T_.mat', [0.012], '2D')

componentAnalysis(1,'eeg_alpha','dstrx', dir_eeg_destrieux + 'subj01-7T/conn_destr_cohi_time_eeg_alpha_subj1-7T_.mat', [0.012], '3D')
componentAnalysis(2,'eeg_alpha','dstrx', dir_eeg_destrieux + 'subj02-7T/conn_destr_cohi_time_eeg_alpha_subj2-7T_.mat', [0.012], '3D')
componentAnalysis(3,'eeg_alpha','dstrx', dir_eeg_destrieux + 'subj03-7T/conn_destr_cohi_time_eeg_alpha_subj3-7T_.mat', [0.012], '3D')
componentAnalysis(4,'eeg_alpha','dstrx', dir_eeg_destrieux + 'subj04-7T/conn_destr_cohi_time_eeg_alpha_subj4-7T_.mat', [0.012], '3D')
componentAnalysis(5,'eeg_alpha','dstrx', dir_eeg_destrieux + 'subj05-7T/conn_destr_cohi_time_eeg_alpha_subj5-7T_.mat', [0.012], '3D')
componentAnalysis(6,'eeg_alpha','dstrx', dir_eeg_destrieux + 'subj06-7T/conn_destr_cohi_time_eeg_alpha_subj6-7T_.mat', [0.012], '3D')
componentAnalysis(7,'eeg_alpha','dstrx', dir_eeg_destrieux + 'subj07-7T/conn_destr_cohi_time_eeg_alpha_subj7-7T_.mat', [0.012], '3D')
componentAnalysis(8,'eeg_alpha','dstrx', dir_eeg_destrieux + 'subj08-7T/conn_destr_cohi_time_eeg_alpha_subj8-7T_.mat', [0.012], '3D')
componentAnalysis(9,'eeg_alpha','dstrx', dir_eeg_destrieux + 'subj09-7T/conn_destr_cohi_time_eeg_alpha_subj9-7T_.mat', [0.012], '3D')

Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant component for time frame:  7
Giant component for time frame:  8
Giant component for time frame:  9
Giant component for time frame:  10
Giant component for time frame:  11
Giant component for time frame:  12
Giant component for time frame:  13
Giant component for time frame:  14
Giant component for time frame:  15
Giant component for time frame:  16
Giant component for time frame:  17
Giant component for time frame:  18
Giant component for time frame:  19
Giant component for time frame:  20
Giant component for time frame:  21
Giant component for time frame:  22
Giant component for time frame:  23
Giant component for time frame:  24
Giant component for time frame:  25
Giant component for time frame:  26
Giant component for time frame:  27
Giant component for time frame:  28
G

Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
Giant component for time frame:  232
Giant component for time frame:  233
Giant component for time frame:  234
Giant component for time frame:  235
Giant component for time frame:  236
Giant component for time frame:  237
Giant component for time frame:  238
Giant component for time frame:  239
Giant component for time frame:  240
Giant component for time frame:  241
Giant component for time frame:  242
Giant component for time frame:  243
Giant component for time frame:  244
Giant component for time frame:  245
Giant component for time frame:  246
Giant component for time frame:  247
Giant component for time frame:  248
Giant component for time frame:  249
Giant component for time frame:  250
Giant component for time frame:  251
Giant component for time frame:  252
G

Giant component for time frame:  49
Giant component for time frame:  50
Giant component for time frame:  51
Giant component for time frame:  52
Giant component for time frame:  53
Giant component for time frame:  54
Giant component for time frame:  55
Giant component for time frame:  56
Giant component for time frame:  57
Giant component for time frame:  58
Giant component for time frame:  59
Giant component for time frame:  60
Giant component for time frame:  61
Giant component for time frame:  62
Giant component for time frame:  63
Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time frame:  74
Giant component for time frame:  75
Giant component for time fra

Giant component for time frame:  272
Giant component for time frame:  273
Giant component for time frame:  274
Giant component for time frame:  275
Giant component for time frame:  276
Giant component for time frame:  277
Giant component for time frame:  278
Giant component for time frame:  279
Giant component for time frame:  280
Giant component for time frame:  281
Giant component for time frame:  282
Giant component for time frame:  283
Giant component for time frame:  284
Giant component for time frame:  285
Giant component for time frame:  286
Giant component for time frame:  287
Giant component for time frame:  288
Giant component for time frame:  289
Giant component for time frame:  290
Giant component for time frame:  291
Giant component for time frame:  292
Giant component for time frame:  293
Giant component for time frame:  294
Giant component for time frame:  295
Giant component for time frame:  296
Giant component for time frame:  297
Giant component for time frame:  298
G

Giant component for time frame:  47
Giant component for time frame:  48
Giant component for time frame:  49
Giant component for time frame:  50
Giant component for time frame:  51
Giant component for time frame:  52
Giant component for time frame:  53
Giant component for time frame:  54
Giant component for time frame:  55
Giant component for time frame:  56
Giant component for time frame:  57
Giant component for time frame:  58
Giant component for time frame:  59
Giant component for time frame:  60
Giant component for time frame:  61
Giant component for time frame:  62
Giant component for time frame:  63
Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time fra

...

Giant component for time frame:  165
Giant component for time frame:  166
Giant component for time frame:  167
Giant component for time frame:  168
Giant component for time frame:  169
Giant component for time frame:  170
Giant component for time frame:  171
Giant component for time frame:  172
Giant component for time frame:  173
Giant component for time frame:  174
Giant component for time frame:  175
Giant component for time frame:  176
Giant component for time frame:  177
Giant component for time frame:  178
Giant component for time frame:  179
Giant component for time frame:  180
Giant component for time frame:  181
Giant component for time frame:  182
Giant component for time frame:  183
Giant component for time frame:  184
Giant component for time frame:  185
Giant component for time frame:  186
Giant component for time frame:  187
Giant component for time frame:  188
Giant component for time frame:  189
Giant component for time frame:  190
Giant component for time frame:  191
G

Giant component for time frame:  387
Giant component for time frame:  388
Giant component for time frame:  389
Giant component for time frame:  390
Giant component for time frame:  391
Giant component for time frame:  392
Giant component for time frame:  393
Giant component for time frame:  394
Giant component for time frame:  395
Giant component for time frame:  396
Giant component for time frame:  397
Giant component for time frame:  398
Giant component for time frame:  399
Giant component for time frame:  400
Giant component for time frame:  401
Giant component for time frame:  402
Giant component for time frame:  403
Giant component for time frame:  404
Giant component for time frame:  405
Giant component for time frame:  406
Giant component for time frame:  407
Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant compone

Giant component for time frame:  205
Giant component for time frame:  206
Giant component for time frame:  207
Giant component for time frame:  208
Giant component for time frame:  209
Giant component for time frame:  210
Giant component for time frame:  211
Giant component for time frame:  212
Giant component for time frame:  213
Giant component for time frame:  214
Giant component for time frame:  215
Giant component for time frame:  216
Giant component for time frame:  217
Giant component for time frame:  218
Giant component for time frame:  219
Giant component for time frame:  220
Giant component for time frame:  221
Giant component for time frame:  222
Giant component for time frame:  223
Giant component for time frame:  224
Giant component for time frame:  225
Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
G

Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time frame:  74
Giant component for time frame:  75
Giant component for time frame:  76
Giant component for time frame:  77
Giant component for time frame:  78
Giant component for time frame:  79
Giant component for time frame:  80
Giant component for time frame:  81
Giant component for time frame:  82
Giant component for time frame:  83
Giant component for time frame:  84
Giant component for time frame:  85
Giant component for time frame:  86
Giant component for time frame:  87
Giant component for time frame:  88
Giant component for time frame:  89
Giant component for time frame:  90
Giant component for time fra

Giant component for time frame:  287
Giant component for time frame:  288
Giant component for time frame:  289
Giant component for time frame:  290
Giant component for time frame:  291
Giant component for time frame:  292
Giant component for time frame:  293
Giant component for time frame:  294
Giant component for time frame:  295
Giant component for time frame:  296
Giant component for time frame:  297
Giant component for time frame:  298
Giant component for time frame:  299
Giant component for time frame:  300
Giant component for time frame:  301
Giant component for time frame:  302
Giant component for time frame:  303
Giant component for time frame:  304
Giant component for time frame:  305
Giant component for time frame:  306
Giant component for time frame:  307
Giant component for time frame:  308
Giant component for time frame:  309
Giant component for time frame:  310
Giant component for time frame:  311
Giant component for time frame:  312
Giant component for time frame:  313
G

In [ ]:
# Destrieux atlas - EEG beta band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_destrieux)):
    
    for file in files:
        
        if 'conn_destr_cohi_time_eeg_beta' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_beta','dstrx',os.path.join(subdir, file), threshold_values)
            s += 1


In [16]:
# Destrieux atlas - EEG beta band

#componentAnalysis(1,'eeg_beta','dstrx', dir_eeg_destrieux + 'subj01-7T/conn_destr_cohi_time_eeg_beta_subj1-7T_.mat', [0.011], '2D')
#componentAnalysis(2,'eeg_beta','dstrx', dir_eeg_destrieux + 'subj02-7T/conn_destr_cohi_time_eeg_beta_subj2-7T_.mat', [0.011], '2D')
#componentAnalysis(3,'eeg_beta','dstrx', dir_eeg_destrieux + 'subj03-7T/conn_destr_cohi_time_eeg_beta_subj3-7T_.mat', [0.011], '2D')
#componentAnalysis(4,'eeg_beta','dstrx', dir_eeg_destrieux + 'subj04-7T/conn_destr_cohi_time_eeg_beta_subj4-7T_.mat', [0.011], '2D')
#componentAnalysis(5,'eeg_beta','dstrx', dir_eeg_destrieux + 'subj05-7T/conn_destr_cohi_time_eeg_beta_subj5-7T_.mat', [0.011], '2D')
#componentAnalysis(6,'eeg_beta','dstrx', dir_eeg_destrieux + 'subj06-7T/conn_destr_cohi_time_eeg_beta_subj6-7T_.mat', [0.011], '2D')
#componentAnalysis(7,'eeg_beta','dstrx', dir_eeg_destrieux + 'subj07-7T/conn_destr_cohi_time_eeg_beta_subj7-7T_.mat', [0.011], '2D')
#componentAnalysis(8,'eeg_beta','dstrx', dir_eeg_destrieux + 'subj08-7T/conn_destr_cohi_time_eeg_beta_subj8-7T_.mat', [0.011], '2D')
#componentAnalysis(9,'eeg_beta','dstrx', dir_eeg_destrieux + 'subj09-7T/conn_destr_cohi_time_eeg_beta_subj9-7T_.mat', [0.011], '2D')

componentAnalysis(1,'eeg_beta','dstrx', dir_eeg_destrieux + 'subj01-7T/conn_destr_cohi_time_eeg_beta_subj1-7T_.mat', [0.011], '3D')
componentAnalysis(2,'eeg_beta','dstrx', dir_eeg_destrieux + 'subj02-7T/conn_destr_cohi_time_eeg_beta_subj2-7T_.mat', [0.011], '3D')
componentAnalysis(3,'eeg_beta','dstrx', dir_eeg_destrieux + 'subj03-7T/conn_destr_cohi_time_eeg_beta_subj3-7T_.mat', [0.011], '3D')
componentAnalysis(4,'eeg_beta','dstrx', dir_eeg_destrieux + 'subj04-7T/conn_destr_cohi_time_eeg_beta_subj4-7T_.mat', [0.011], '3D')
componentAnalysis(5,'eeg_beta','dstrx', dir_eeg_destrieux + 'subj05-7T/conn_destr_cohi_time_eeg_beta_subj5-7T_.mat', [0.011], '3D')
componentAnalysis(6,'eeg_beta','dstrx', dir_eeg_destrieux + 'subj06-7T/conn_destr_cohi_time_eeg_beta_subj6-7T_.mat', [0.011], '3D')
componentAnalysis(7,'eeg_beta','dstrx', dir_eeg_destrieux + 'subj07-7T/conn_destr_cohi_time_eeg_beta_subj7-7T_.mat', [0.011], '3D')
componentAnalysis(8,'eeg_beta','dstrx', dir_eeg_destrieux + 'subj08-7T/conn_destr_cohi_time_eeg_beta_subj8-7T_.mat', [0.011], '3D')
componentAnalysis(9,'eeg_beta','dstrx', dir_eeg_destrieux + 'subj09-7T/conn_destr_cohi_time_eeg_beta_subj9-7T_.mat', [0.011], '3D')

Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant component for time frame:  7
Giant component for time frame:  8
Giant component for time frame:  9
Giant component for time frame:  10
Giant component for time frame:  11
Giant component for time frame:  12
Giant component for time frame:  13
Giant component for time frame:  14
Giant component for time frame:  15
Giant component for time frame:  16
Giant component for time frame:  17
Giant component for time frame:  18
Giant component for time frame:  19
Giant component for time frame:  20
Giant component for time frame:  21
Giant component for time frame:  22
Giant component for time frame:  23
Giant component for time frame:  24
Giant component for time frame:  25
Giant component for time frame:  26
Giant component for time frame:  27
Giant component for time frame:  28
G

Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
Giant component for time frame:  232
Giant component for time frame:  233
Giant component for time frame:  234
Giant component for time frame:  235
Giant component for time frame:  236
Giant component for time frame:  237
Giant component for time frame:  238
Giant component for time frame:  239
Giant component for time frame:  240
Giant component for time frame:  241
Giant component for time frame:  242
Giant component for time frame:  243
Giant component for time frame:  244
Giant component for time frame:  245
Giant component for time frame:  246
Giant component for time frame:  247
Giant component for time frame:  248
Giant component for time frame:  249
Giant component for time frame:  250
Giant component for time frame:  251
Giant component for time frame:  252
G

Giant component for time frame:  49
Giant component for time frame:  50
Giant component for time frame:  51
Giant component for time frame:  52
Giant component for time frame:  53
Giant component for time frame:  54
Giant component for time frame:  55
Giant component for time frame:  56
Giant component for time frame:  57
Giant component for time frame:  58
Giant component for time frame:  59
Giant component for time frame:  60
Giant component for time frame:  61
Giant component for time frame:  62
Giant component for time frame:  63
Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time frame:  74
Giant component for time frame:  75
Giant component for time fra

Giant component for time frame:  272
Giant component for time frame:  273
Giant component for time frame:  274
Giant component for time frame:  275
Giant component for time frame:  276
Giant component for time frame:  277
Giant component for time frame:  278
Giant component for time frame:  279
Giant component for time frame:  280
Giant component for time frame:  281
Giant component for time frame:  282
Giant component for time frame:  283
Giant component for time frame:  284
Giant component for time frame:  285
Giant component for time frame:  286
Giant component for time frame:  287
Giant component for time frame:  288
Giant component for time frame:  289
Giant component for time frame:  290
Giant component for time frame:  291
Giant component for time frame:  292
Giant component for time frame:  293
Giant component for time frame:  294
Giant component for time frame:  295
Giant component for time frame:  296
Giant component for time frame:  297
Giant component for time frame:  298
G

Giant component for time frame:  47
Giant component for time frame:  48
Giant component for time frame:  49
Giant component for time frame:  50
Giant component for time frame:  51
Giant component for time frame:  52
Giant component for time frame:  53
Giant component for time frame:  54
Giant component for time frame:  55
Giant component for time frame:  56
Giant component for time frame:  57
Giant component for time frame:  58
Giant component for time frame:  59
Giant component for time frame:  60
Giant component for time frame:  61
Giant component for time frame:  62
Giant component for time frame:  63
Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time fra

...

Giant component for time frame:  165
Giant component for time frame:  166
Giant component for time frame:  167
Giant component for time frame:  168
Giant component for time frame:  169
Giant component for time frame:  170
Giant component for time frame:  171
Giant component for time frame:  172
Giant component for time frame:  173
Giant component for time frame:  174
Giant component for time frame:  175
Giant component for time frame:  176
Giant component for time frame:  177
Giant component for time frame:  178
Giant component for time frame:  179
Giant component for time frame:  180
Giant component for time frame:  181
Giant component for time frame:  182
Giant component for time frame:  183
Giant component for time frame:  184
Giant component for time frame:  185
Giant component for time frame:  186
Giant component for time frame:  187
Giant component for time frame:  188
Giant component for time frame:  189
Giant component for time frame:  190
Giant component for time frame:  191
G

Giant component for time frame:  387
Giant component for time frame:  388
Giant component for time frame:  389
Giant component for time frame:  390
Giant component for time frame:  391
Giant component for time frame:  392
Giant component for time frame:  393
Giant component for time frame:  394
Giant component for time frame:  395
Giant component for time frame:  396
Giant component for time frame:  397
Giant component for time frame:  398
Giant component for time frame:  399
Giant component for time frame:  400
Giant component for time frame:  401
Giant component for time frame:  402
Giant component for time frame:  403
Giant component for time frame:  404
Giant component for time frame:  405
Giant component for time frame:  406
Giant component for time frame:  407
Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant compone

Giant component for time frame:  205
Giant component for time frame:  206
Giant component for time frame:  207
Giant component for time frame:  208
Giant component for time frame:  209
Giant component for time frame:  210
Giant component for time frame:  211
Giant component for time frame:  212
Giant component for time frame:  213
Giant component for time frame:  214
Giant component for time frame:  215
Giant component for time frame:  216
Giant component for time frame:  217
Giant component for time frame:  218
Giant component for time frame:  219
Giant component for time frame:  220
Giant component for time frame:  221
Giant component for time frame:  222
Giant component for time frame:  223
Giant component for time frame:  224
Giant component for time frame:  225
Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
G

Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time frame:  74
Giant component for time frame:  75
Giant component for time frame:  76
Giant component for time frame:  77
Giant component for time frame:  78
Giant component for time frame:  79
Giant component for time frame:  80
Giant component for time frame:  81
Giant component for time frame:  82
Giant component for time frame:  83
Giant component for time frame:  84
Giant component for time frame:  85
Giant component for time frame:  86
Giant component for time frame:  87
Giant component for time frame:  88
Giant component for time frame:  89
Giant component for time frame:  90
Giant component for time fra

Giant component for time frame:  287
Giant component for time frame:  288
Giant component for time frame:  289
Giant component for time frame:  290
Giant component for time frame:  291
Giant component for time frame:  292
Giant component for time frame:  293
Giant component for time frame:  294
Giant component for time frame:  295
Giant component for time frame:  296
Giant component for time frame:  297
Giant component for time frame:  298
Giant component for time frame:  299
Giant component for time frame:  300
Giant component for time frame:  301
Giant component for time frame:  302
Giant component for time frame:  303
Giant component for time frame:  304
Giant component for time frame:  305
Giant component for time frame:  306
Giant component for time frame:  307
Giant component for time frame:  308
Giant component for time frame:  309
Giant component for time frame:  310
Giant component for time frame:  311
Giant component for time frame:  312
Giant component for time frame:  313
G

In [9]:
# Destrieux atlas - EEG delta band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_destrieux)):
    
    for file in files:
        
        if 'conn_destr_cohi_time_eeg_delta' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_delta','dstrx',os.path.join(subdir, file), threshold_values)
            s += 1


Subject 1, with eeg_delta and using dstrx
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 15
Number of nodes of the second biggest component: 10
___________________________________________________________________________
Proportion kept: 0.8
Number of edges of thresholded graph: 87
Average degree of thresholded graph:  1.1756756756756757
Number of nodes of the giant component: 30
Number of nodes of the second biggest component: 11
___________________________________________________________________________
Proportion kept: 1.0
Number of edges of thresholded graph: 109
Average degree of thresholded graph:  1.472972972972973
Number of nodes of the giant component: 38
Number of nodes of the second biggest component: 15
___________________________________________________________________________
Proportion kept: 1.2
Number of edges of thresholded graph: 131
Average degree of thresho

Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Subject 3, with eeg_delta and using dstrx
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 25
Number of nodes of the second biggest component: 6
___________________________________________________________________________
Proportion kept: 0.8
Number of edges of thresholded graph: 87
Average degree of thresholde

Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Subject 5, with eeg_delta and using dstrx
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 19
Number of nodes of the second biggest component: 8
___________________________________________________________________________
Proportion kept: 0.8
Number of edges of thresholded graph: 87
Average degree of thresholde

Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Subject 7, with eeg_delta and using dstrx
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 11
Number of nodes of the second biggest component: 8
___________________________________________________________________________
Proportion kept: 0.8
Number of edges of thresholded graph: 87
Average degree of thresholde

Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Subject 9, with eeg_delta and using dstrx
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 17
Number of nodes of the second biggest component: 10
___________________________________________________________________________
Proportion kept: 0.8
Number of edges of thresholded graph: 87
Average degree of threshold

In [17]:
# Destrieux atlas - EEG delta band

#componentAnalysis(1,'eeg_delta','dstrx', dir_eeg_destrieux + 'subj01-7T/conn_destr_cohi_time_eeg_delta_subj1-7T_.mat', [0.011], '2D')
#componentAnalysis(2,'eeg_delta','dstrx', dir_eeg_destrieux + 'subj02-7T/conn_destr_cohi_time_eeg_delta_subj2-7T_.mat', [0.011], '2D')
#componentAnalysis(3,'eeg_delta','dstrx', dir_eeg_destrieux + 'subj03-7T/conn_destr_cohi_time_eeg_delta_subj3-7T_.mat', [0.011], '2D')
#componentAnalysis(4,'eeg_delta','dstrx', dir_eeg_destrieux + 'subj04-7T/conn_destr_cohi_time_eeg_delta_subj4-7T_.mat', [0.011], '2D')
#componentAnalysis(5,'eeg_delta','dstrx', dir_eeg_destrieux + 'subj05-7T/conn_destr_cohi_time_eeg_delta_subj5-7T_.mat', [0.011], '2D')
#componentAnalysis(6,'eeg_delta','dstrx', dir_eeg_destrieux + 'subj06-7T/conn_destr_cohi_time_eeg_delta_subj6-7T_.mat', [0.011], '2D')
#componentAnalysis(7,'eeg_delta','dstrx', dir_eeg_destrieux + 'subj07-7T/conn_destr_cohi_time_eeg_delta_subj7-7T_.mat', [0.011], '2D')
#componentAnalysis(8,'eeg_delta','dstrx', dir_eeg_destrieux + 'subj08-7T/conn_destr_cohi_time_eeg_delta_subj8-7T_.mat', [0.011], '2D')
#componentAnalysis(9,'eeg_delta','dstrx', dir_eeg_destrieux + 'subj09-7T/conn_destr_cohi_time_eeg_delta_subj9-7T_.mat', [0.011], '2D')

componentAnalysis(1,'eeg_delta','dstrx', dir_eeg_destrieux + 'subj01-7T/conn_destr_cohi_time_eeg_delta_subj1-7T_.mat', [0.011], '3D')
componentAnalysis(2,'eeg_delta','dstrx', dir_eeg_destrieux + 'subj02-7T/conn_destr_cohi_time_eeg_delta_subj2-7T_.mat', [0.011], '3D')
componentAnalysis(3,'eeg_delta','dstrx', dir_eeg_destrieux + 'subj03-7T/conn_destr_cohi_time_eeg_delta_subj3-7T_.mat', [0.011], '3D')
componentAnalysis(4,'eeg_delta','dstrx', dir_eeg_destrieux + 'subj04-7T/conn_destr_cohi_time_eeg_delta_subj4-7T_.mat', [0.011], '3D')
componentAnalysis(5,'eeg_delta','dstrx', dir_eeg_destrieux + 'subj05-7T/conn_destr_cohi_time_eeg_delta_subj5-7T_.mat', [0.011], '3D')
componentAnalysis(6,'eeg_delta','dstrx', dir_eeg_destrieux + 'subj06-7T/conn_destr_cohi_time_eeg_delta_subj6-7T_.mat', [0.011], '3D')
componentAnalysis(7,'eeg_delta','dstrx', dir_eeg_destrieux + 'subj07-7T/conn_destr_cohi_time_eeg_delta_subj7-7T_.mat', [0.011], '3D')
componentAnalysis(8,'eeg_delta','dstrx', dir_eeg_destrieux + 'subj08-7T/conn_destr_cohi_time_eeg_delta_subj8-7T_.mat', [0.011], '3D')
componentAnalysis(9,'eeg_delta','dstrx', dir_eeg_destrieux + 'subj09-7T/conn_destr_cohi_time_eeg_delta_subj9-7T_.mat', [0.011], '3D')

Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant component for time frame:  7
Giant component for time frame:  8
Giant component for time frame:  9
Giant component for time frame:  10
Giant component for time frame:  11
Giant component for time frame:  12
Giant component for time frame:  13
Giant component for time frame:  14
Giant component for time frame:  15
Giant component for time frame:  16
Giant component for time frame:  17
Giant component for time frame:  18
Giant component for time frame:  19
Giant component for time frame:  20
Giant component for time frame:  21
Giant component for time frame:  22
Giant component for time frame:  23
Giant component for time frame:  24
Giant component for time frame:  25
Giant component for time frame:  26
Giant component for time frame:  27
Giant component for time frame:  28
G

Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
Giant component for time frame:  232
Giant component for time frame:  233
Giant component for time frame:  234
Giant component for time frame:  235
Giant component for time frame:  236
Giant component for time frame:  237
Giant component for time frame:  238
Giant component for time frame:  239
Giant component for time frame:  240
Giant component for time frame:  241
Giant component for time frame:  242
Giant component for time frame:  243
Giant component for time frame:  244
Giant component for time frame:  245
Giant component for time frame:  246
Giant component for time frame:  247
Giant component for time frame:  248
Giant component for time frame:  249
Giant component for time frame:  250
Giant component for time frame:  251
Giant component for time frame:  252
G

Giant component for time frame:  49
Giant component for time frame:  50
Giant component for time frame:  51
Giant component for time frame:  52
Giant component for time frame:  53
Giant component for time frame:  54
Giant component for time frame:  55
Giant component for time frame:  56
Giant component for time frame:  57
Giant component for time frame:  58
Giant component for time frame:  59
Giant component for time frame:  60
Giant component for time frame:  61
Giant component for time frame:  62
Giant component for time frame:  63
Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time frame:  74
Giant component for time frame:  75
Giant component for time fra

Giant component for time frame:  272
Giant component for time frame:  273
Giant component for time frame:  274
Giant component for time frame:  275
Giant component for time frame:  276
Giant component for time frame:  277
Giant component for time frame:  278
Giant component for time frame:  279
Giant component for time frame:  280
Giant component for time frame:  281
Giant component for time frame:  282
Giant component for time frame:  283
Giant component for time frame:  284
Giant component for time frame:  285
Giant component for time frame:  286
Giant component for time frame:  287
Giant component for time frame:  288
Giant component for time frame:  289
Giant component for time frame:  290
Giant component for time frame:  291
Giant component for time frame:  292
Giant component for time frame:  293
Giant component for time frame:  294
Giant component for time frame:  295
Giant component for time frame:  296
Giant component for time frame:  297
Giant component for time frame:  298
G

Giant component for time frame:  47
Giant component for time frame:  48
Giant component for time frame:  49
Giant component for time frame:  50
Giant component for time frame:  51
Giant component for time frame:  52
Giant component for time frame:  53
Giant component for time frame:  54
Giant component for time frame:  55
Giant component for time frame:  56
Giant component for time frame:  57
Giant component for time frame:  58
Giant component for time frame:  59
Giant component for time frame:  60
Giant component for time frame:  61
Giant component for time frame:  62
Giant component for time frame:  63
Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time fra

...

Giant component for time frame:  165
Giant component for time frame:  166
Giant component for time frame:  167
Giant component for time frame:  168
Giant component for time frame:  169
Giant component for time frame:  170
Giant component for time frame:  171
Giant component for time frame:  172
Giant component for time frame:  173
Giant component for time frame:  174
Giant component for time frame:  175
Giant component for time frame:  176
Giant component for time frame:  177
Giant component for time frame:  178
Giant component for time frame:  179
Giant component for time frame:  180
Giant component for time frame:  181
Giant component for time frame:  182
Giant component for time frame:  183
Giant component for time frame:  184
Giant component for time frame:  185
Giant component for time frame:  186
Giant component for time frame:  187
Giant component for time frame:  188
Giant component for time frame:  189
Giant component for time frame:  190
Giant component for time frame:  191
G

Giant component for time frame:  387
Giant component for time frame:  388
Giant component for time frame:  389
Giant component for time frame:  390
Giant component for time frame:  391
Giant component for time frame:  392
Giant component for time frame:  393
Giant component for time frame:  394
Giant component for time frame:  395
Giant component for time frame:  396
Giant component for time frame:  397
Giant component for time frame:  398
Giant component for time frame:  399
Giant component for time frame:  400
Giant component for time frame:  401
Giant component for time frame:  402
Giant component for time frame:  403
Giant component for time frame:  404
Giant component for time frame:  405
Giant component for time frame:  406
Giant component for time frame:  407
Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant compone

Giant component for time frame:  205
Giant component for time frame:  206
Giant component for time frame:  207
Giant component for time frame:  208
Giant component for time frame:  209
Giant component for time frame:  210
Giant component for time frame:  211
Giant component for time frame:  212
Giant component for time frame:  213
Giant component for time frame:  214
Giant component for time frame:  215
Giant component for time frame:  216
Giant component for time frame:  217
Giant component for time frame:  218
Giant component for time frame:  219
Giant component for time frame:  220
Giant component for time frame:  221
Giant component for time frame:  222
Giant component for time frame:  223
Giant component for time frame:  224
Giant component for time frame:  225
Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
G

Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time frame:  74
Giant component for time frame:  75
Giant component for time frame:  76
Giant component for time frame:  77
Giant component for time frame:  78
Giant component for time frame:  79
Giant component for time frame:  80
Giant component for time frame:  81
Giant component for time frame:  82
Giant component for time frame:  83
Giant component for time frame:  84
Giant component for time frame:  85
Giant component for time frame:  86
Giant component for time frame:  87
Giant component for time frame:  88
Giant component for time frame:  89
Giant component for time frame:  90
Giant component for time fra

Giant component for time frame:  287
Giant component for time frame:  288
Giant component for time frame:  289
Giant component for time frame:  290
Giant component for time frame:  291
Giant component for time frame:  292
Giant component for time frame:  293
Giant component for time frame:  294
Giant component for time frame:  295
Giant component for time frame:  296
Giant component for time frame:  297
Giant component for time frame:  298
Giant component for time frame:  299
Giant component for time frame:  300
Giant component for time frame:  301
Giant component for time frame:  302
Giant component for time frame:  303
Giant component for time frame:  304
Giant component for time frame:  305
Giant component for time frame:  306
Giant component for time frame:  307
Giant component for time frame:  308
Giant component for time frame:  309
Giant component for time frame:  310
Giant component for time frame:  311
Giant component for time frame:  312
Giant component for time frame:  313
G

In [10]:
# Destrieux atlas - EEG gamma band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_destrieux)):
    
    for file in files:
        
        if 'conn_destr_cohi_time_eeg_gamma' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_gamma','dstrx',os.path.join(subdir, file), threshold_values)
            s += 1


Subject 1, with eeg_gamma and using dstrx
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 25
Number of nodes of the second biggest component: 11
___________________________________________________________________________
Proportion kept: 0.8
Number of edges of thresholded graph: 87
Average degree of thresholded graph:  1.1756756756756757
Number of nodes of the giant component: 27
Number of nodes of the second biggest component: 12
___________________________________________________________________________
Proportion kept: 1.0
Number of edges of thresholded graph: 109
Average degree of thresholded graph:  1.472972972972973
Number of nodes of the giant component: 32
Number of nodes of the second biggest component: 12
___________________________________________________________________________
Proportion kept: 1.2
Number of edges of thresholded graph: 131
Average degree of thresho

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Subject 3, with eeg_gamma and using dstrx
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 20
Number of nodes of the second biggest component: 2
___________________________________________________________________

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Subject 5, with eeg_gamma and using dstrx
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 17
Number of nodes of the second biggest component: 4
___________________________________________________________________

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Subject 7, with eeg_gamma and using dstrx
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 29
Number of nodes of the second biggest component: 2
___________________________________________________________________

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Subject 9, with eeg_gamma and using dstrx
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 33
Number of nodes of the second biggest component: 3
___________________________________________________________________

In [18]:
# Destrieux atlas - EEG gamma band

#componentAnalysis(1,'eeg_gamma','dstrx', dir_eeg_destrieux + 'subj01-7T/conn_destr_cohi_time_eeg_gamma_subj1-7T_.mat', [0.011], '2D')
#componentAnalysis(2,'eeg_gamma','dstrx', dir_eeg_destrieux + 'subj02-7T/conn_destr_cohi_time_eeg_gamma_subj2-7T_.mat', [0.011], '2D')
#componentAnalysis(3,'eeg_gamma','dstrx', dir_eeg_destrieux + 'subj03-7T/conn_destr_cohi_time_eeg_gamma_subj3-7T_.mat', [0.011], '2D')
#componentAnalysis(4,'eeg_gamma','dstrx', dir_eeg_destrieux + 'subj04-7T/conn_destr_cohi_time_eeg_gamma_subj4-7T_.mat', [0.011], '2D')
#componentAnalysis(5,'eeg_gamma','dstrx', dir_eeg_destrieux + 'subj05-7T/conn_destr_cohi_time_eeg_gamma_subj5-7T_.mat', [0.011], '2D')
#componentAnalysis(6,'eeg_gamma','dstrx', dir_eeg_destrieux + 'subj06-7T/conn_destr_cohi_time_eeg_gamma_subj6-7T_.mat', [0.011], '2D')
#componentAnalysis(7,'eeg_gamma','dstrx', dir_eeg_destrieux + 'subj07-7T/conn_destr_cohi_time_eeg_gamma_subj7-7T_.mat', [0.011], '2D')
#componentAnalysis(8,'eeg_gamma','dstrx', dir_eeg_destrieux + 'subj08-7T/conn_destr_cohi_time_eeg_gamma_subj8-7T_.mat', [0.011], '2D')
#componentAnalysis(9,'eeg_gamma','dstrx', dir_eeg_destrieux + 'subj09-7T/conn_destr_cohi_time_eeg_gamma_subj9-7T_.mat', [0.011], '2D')

componentAnalysis(1,'eeg_gamma','dstrx', dir_eeg_destrieux + 'subj01-7T/conn_destr_cohi_time_eeg_gamma_subj1-7T_.mat', [0.011], '3D')
componentAnalysis(2,'eeg_gamma','dstrx', dir_eeg_destrieux + 'subj02-7T/conn_destr_cohi_time_eeg_gamma_subj2-7T_.mat', [0.011], '3D')
componentAnalysis(3,'eeg_gamma','dstrx', dir_eeg_destrieux + 'subj03-7T/conn_destr_cohi_time_eeg_gamma_subj3-7T_.mat', [0.011], '3D')
componentAnalysis(4,'eeg_gamma','dstrx', dir_eeg_destrieux + 'subj04-7T/conn_destr_cohi_time_eeg_gamma_subj4-7T_.mat', [0.011], '3D')
componentAnalysis(5,'eeg_gamma','dstrx', dir_eeg_destrieux + 'subj05-7T/conn_destr_cohi_time_eeg_gamma_subj5-7T_.mat', [0.011], '3D')
componentAnalysis(6,'eeg_gamma','dstrx', dir_eeg_destrieux + 'subj06-7T/conn_destr_cohi_time_eeg_gamma_subj6-7T_.mat', [0.011], '3D')
componentAnalysis(7,'eeg_gamma','dstrx', dir_eeg_destrieux + 'subj07-7T/conn_destr_cohi_time_eeg_gamma_subj7-7T_.mat', [0.011], '3D')
componentAnalysis(8,'eeg_gamma','dstrx', dir_eeg_destrieux + 'subj08-7T/conn_destr_cohi_time_eeg_gamma_subj8-7T_.mat', [0.011], '3D')
componentAnalysis(9,'eeg_gamma','dstrx', dir_eeg_destrieux + 'subj09-7T/conn_destr_cohi_time_eeg_gamma_subj9-7T_.mat', [0.011], '3D')


Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant component for time frame:  7
Giant component for time frame:  8
Giant component for time frame:  9
Giant component for time frame:  10
Giant component for time frame:  11
Giant component for time frame:  12
Giant component for time frame:  13
Giant component for time frame:  14
Giant component for time frame:  15
Giant component for time frame:  16
Giant component for time frame:  17
Giant component for time frame:  18
Giant component for time frame:  19
Giant component for time frame:  20
Giant component for time frame:  21
Giant component for time frame:  22
Giant component for time frame:  23
Giant component for time frame:  24
Giant component for time frame:  25
Giant component for time frame:  26
Giant component for time frame:  27
Giant component for time frame:  28
G

Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
Giant component for time frame:  232
Giant component for time frame:  233
Giant component for time frame:  234
Giant component for time frame:  235
Giant component for time frame:  236
Giant component for time frame:  237
Giant component for time frame:  238
Giant component for time frame:  239
Giant component for time frame:  240
Giant component for time frame:  241
Giant component for time frame:  242
Giant component for time frame:  243
Giant component for time frame:  244
Giant component for time frame:  245
Giant component for time frame:  246
Giant component for time frame:  247
Giant component for time frame:  248
Giant component for time frame:  249
Giant component for time frame:  250
Giant component for time frame:  251
Giant component for time frame:  252
G

Giant component for time frame:  49
Giant component for time frame:  50
Giant component for time frame:  51
Giant component for time frame:  52
Giant component for time frame:  53
Giant component for time frame:  54
Giant component for time frame:  55
Giant component for time frame:  56
Giant component for time frame:  57
Giant component for time frame:  58
Giant component for time frame:  59
Giant component for time frame:  60
Giant component for time frame:  61
Giant component for time frame:  62
Giant component for time frame:  63
Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time frame:  74
Giant component for time frame:  75
Giant component for time fra

Giant component for time frame:  272
Giant component for time frame:  273
Giant component for time frame:  274
Giant component for time frame:  275
Giant component for time frame:  276
Giant component for time frame:  277
Giant component for time frame:  278
Giant component for time frame:  279
Giant component for time frame:  280
Giant component for time frame:  281
Giant component for time frame:  282
Giant component for time frame:  283
Giant component for time frame:  284
Giant component for time frame:  285
Giant component for time frame:  286
Giant component for time frame:  287
Giant component for time frame:  288
Giant component for time frame:  289
Giant component for time frame:  290
Giant component for time frame:  291
Giant component for time frame:  292
Giant component for time frame:  293
Giant component for time frame:  294
Giant component for time frame:  295
Giant component for time frame:  296
Giant component for time frame:  297
Giant component for time frame:  298
G

Giant component for time frame:  47
Giant component for time frame:  48
Giant component for time frame:  49
Giant component for time frame:  50
Giant component for time frame:  51
Giant component for time frame:  52
Giant component for time frame:  53
Giant component for time frame:  54
Giant component for time frame:  55
Giant component for time frame:  56
Giant component for time frame:  57
Giant component for time frame:  58
Giant component for time frame:  59
Giant component for time frame:  60
Giant component for time frame:  61
Giant component for time frame:  62
Giant component for time frame:  63
Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time fra

...

Giant component for time frame:  165
Giant component for time frame:  166
Giant component for time frame:  167
Giant component for time frame:  168
Giant component for time frame:  169
Giant component for time frame:  170
Giant component for time frame:  171
Giant component for time frame:  172
Giant component for time frame:  173
Giant component for time frame:  174
Giant component for time frame:  175
Giant component for time frame:  176
Giant component for time frame:  177
Giant component for time frame:  178
Giant component for time frame:  179
Giant component for time frame:  180
Giant component for time frame:  181
Giant component for time frame:  182
Giant component for time frame:  183
Giant component for time frame:  184
Giant component for time frame:  185
Giant component for time frame:  186
Giant component for time frame:  187
Giant component for time frame:  188
Giant component for time frame:  189
Giant component for time frame:  190
Giant component for time frame:  191
G

Giant component for time frame:  387
Giant component for time frame:  388
Giant component for time frame:  389
Giant component for time frame:  390
Giant component for time frame:  391
Giant component for time frame:  392
Giant component for time frame:  393
Giant component for time frame:  394
Giant component for time frame:  395
Giant component for time frame:  396
Giant component for time frame:  397
Giant component for time frame:  398
Giant component for time frame:  399
Giant component for time frame:  400
Giant component for time frame:  401
Giant component for time frame:  402
Giant component for time frame:  403
Giant component for time frame:  404
Giant component for time frame:  405
Giant component for time frame:  406
Giant component for time frame:  407
Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant compone

Giant component for time frame:  205
Giant component for time frame:  206
Giant component for time frame:  207
Giant component for time frame:  208
Giant component for time frame:  209
Giant component for time frame:  210
Giant component for time frame:  211
Giant component for time frame:  212
Giant component for time frame:  213
Giant component for time frame:  214
Giant component for time frame:  215
Giant component for time frame:  216
Giant component for time frame:  217
Giant component for time frame:  218
Giant component for time frame:  219
Giant component for time frame:  220
Giant component for time frame:  221
Giant component for time frame:  222
Giant component for time frame:  223
Giant component for time frame:  224
Giant component for time frame:  225
Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
G

Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time frame:  74
Giant component for time frame:  75
Giant component for time frame:  76
Giant component for time frame:  77
Giant component for time frame:  78
Giant component for time frame:  79
Giant component for time frame:  80
Giant component for time frame:  81
Giant component for time frame:  82
Giant component for time frame:  83
Giant component for time frame:  84
Giant component for time frame:  85
Giant component for time frame:  86
Giant component for time frame:  87
Giant component for time frame:  88
Giant component for time frame:  89
Giant component for time frame:  90
Giant component for time fra

Giant component for time frame:  287
Giant component for time frame:  288
Giant component for time frame:  289
Giant component for time frame:  290
Giant component for time frame:  291
Giant component for time frame:  292
Giant component for time frame:  293
Giant component for time frame:  294
Giant component for time frame:  295
Giant component for time frame:  296
Giant component for time frame:  297
Giant component for time frame:  298
Giant component for time frame:  299
Giant component for time frame:  300
Giant component for time frame:  301
Giant component for time frame:  302
Giant component for time frame:  303
Giant component for time frame:  304
Giant component for time frame:  305
Giant component for time frame:  306
Giant component for time frame:  307
Giant component for time frame:  308
Giant component for time frame:  309
Giant component for time frame:  310
Giant component for time frame:  311
Giant component for time frame:  312
Giant component for time frame:  313
G

In [11]:
# Destrieux atlas - EEG theta band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_destrieux)):
    
    for file in files:
        
        if 'conn_destr_cohi_time_eeg_theta' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_theta','dstrx',os.path.join(subdir, file), threshold_values)
            s += 1


Subject 1, with eeg_theta and using dstrx
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 27
Number of nodes of the second biggest component: 5
___________________________________________________________________________
Proportion kept: 0.8
Number of edges of thresholded graph: 87
Average degree of thresholded graph:  1.1756756756756757
Number of nodes of the giant component: 43
Number of nodes of the second biggest component: 4
___________________________________________________________________________
Proportion kept: 1.0
Number of edges of thresholded graph: 109
Average degree of thresholded graph:  1.472972972972973
Number of nodes of the giant component: 52
Number of nodes of the second biggest component: 4
___________________________________________________________________________
Proportion kept: 1.2
Number of edges of thresholded graph: 131
Average degree of thresholde

Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Subject 3, with eeg_theta and using dstrx
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 25
Number of nodes of the second biggest component: 4
___________________________________________________________________________
Proportion kept: 0.8
Number of edges of thresholded graph: 87
Average degree of thresholde

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Subject 5, with eeg_theta and using dstrx
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 16
Number of nodes of the second biggest component: 15
__________________________________________________________________

Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Subject 7, with eeg_theta and using dstrx
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 16
Number of nodes of the second biggest component: 7
___________________________________________________________________________
Proportion kept: 0.8
Number of edges of thresholded graph: 87
Average degree of thresholde

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Subject 9, with eeg_theta and using dstrx
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 7
Number of nodes of the second biggest component: 7
____________________________________________________________________

In [19]:
# Destrieux atlas - EEG theta band

#componentAnalysis(1,'eeg_theta','dstrx', dir_eeg_destrieux + 'subj01-7T/conn_destr_cohi_time_eeg_theta_subj1-7T_.mat', [0.012], '2D')
#componentAnalysis(2,'eeg_theta','dstrx', dir_eeg_destrieux + 'subj02-7T/conn_destr_cohi_time_eeg_theta_subj2-7T_.mat', [0.012], '2D')
#componentAnalysis(3,'eeg_theta','dstrx', dir_eeg_destrieux + 'subj03-7T/conn_destr_cohi_time_eeg_theta_subj3-7T_.mat', [0.012], '2D')
#componentAnalysis(4,'eeg_theta','dstrx', dir_eeg_destrieux + 'subj04-7T/conn_destr_cohi_time_eeg_theta_subj4-7T_.mat', [0.012], '2D')
#componentAnalysis(5,'eeg_theta','dstrx', dir_eeg_destrieux + 'subj05-7T/conn_destr_cohi_time_eeg_theta_subj5-7T_.mat', [0.012], '2D')
#componentAnalysis(6,'eeg_theta','dstrx', dir_eeg_destrieux + 'subj06-7T/conn_destr_cohi_time_eeg_theta_subj6-7T_.mat', [0.012], '2D')
#componentAnalysis(7,'eeg_theta','dstrx', dir_eeg_destrieux + 'subj07-7T/conn_destr_cohi_time_eeg_theta_subj7-7T_.mat', [0.012], '2D')
#componentAnalysis(8,'eeg_theta','dstrx', dir_eeg_destrieux + 'subj08-7T/conn_destr_cohi_time_eeg_theta_subj8-7T_.mat', [0.012], '2D')
#componentAnalysis(9,'eeg_theta','dstrx', dir_eeg_destrieux + 'subj09-7T/conn_destr_cohi_time_eeg_theta_subj9-7T_.mat', [0.012], '2D')

componentAnalysis(1,'eeg_theta','dstrx', dir_eeg_destrieux + 'subj01-7T/conn_destr_cohi_time_eeg_theta_subj1-7T_.mat', [0.012], '3D')
componentAnalysis(2,'eeg_theta','dstrx', dir_eeg_destrieux + 'subj02-7T/conn_destr_cohi_time_eeg_theta_subj2-7T_.mat', [0.012], '3D')
componentAnalysis(3,'eeg_theta','dstrx', dir_eeg_destrieux + 'subj03-7T/conn_destr_cohi_time_eeg_theta_subj3-7T_.mat', [0.012], '3D')
componentAnalysis(4,'eeg_theta','dstrx', dir_eeg_destrieux + 'subj04-7T/conn_destr_cohi_time_eeg_theta_subj4-7T_.mat', [0.012], '3D')
componentAnalysis(5,'eeg_theta','dstrx', dir_eeg_destrieux + 'subj05-7T/conn_destr_cohi_time_eeg_theta_subj5-7T_.mat', [0.012], '3D')
componentAnalysis(6,'eeg_theta','dstrx', dir_eeg_destrieux + 'subj06-7T/conn_destr_cohi_time_eeg_theta_subj6-7T_.mat', [0.012], '3D')
componentAnalysis(7,'eeg_theta','dstrx', dir_eeg_destrieux + 'subj07-7T/conn_destr_cohi_time_eeg_theta_subj7-7T_.mat', [0.012], '3D')
componentAnalysis(8,'eeg_theta','dstrx', dir_eeg_destrieux + 'subj08-7T/conn_destr_cohi_time_eeg_theta_subj8-7T_.mat', [0.012], '3D')
componentAnalysis(9,'eeg_theta','dstrx', dir_eeg_destrieux + 'subj09-7T/conn_destr_cohi_time_eeg_theta_subj9-7T_.mat', [0.012], '3D')

Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant component for time frame:  7
Giant component for time frame:  8
Giant component for time frame:  9
Giant component for time frame:  10
Giant component for time frame:  11
Giant component for time frame:  12
Giant component for time frame:  13
Giant component for time frame:  14
Giant component for time frame:  15
Giant component for time frame:  16
Giant component for time frame:  17
Giant component for time frame:  18
Giant component for time frame:  19
Giant component for time frame:  20
Giant component for time frame:  21
Giant component for time frame:  22
Giant component for time frame:  23
Giant component for time frame:  24
Giant component for time frame:  25
Giant component for time frame:  26
Giant component for time frame:  27
Giant component for time frame:  28
G

Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
Giant component for time frame:  232
Giant component for time frame:  233
Giant component for time frame:  234
Giant component for time frame:  235
Giant component for time frame:  236
Giant component for time frame:  237
Giant component for time frame:  238
Giant component for time frame:  239
Giant component for time frame:  240
Giant component for time frame:  241
Giant component for time frame:  242
Giant component for time frame:  243
Giant component for time frame:  244
Giant component for time frame:  245
Giant component for time frame:  246
Giant component for time frame:  247
Giant component for time frame:  248
Giant component for time frame:  249
Giant component for time frame:  250
Giant component for time frame:  251
Giant component for time frame:  252
G

Giant component for time frame:  49
Giant component for time frame:  50
Giant component for time frame:  51
Giant component for time frame:  52
Giant component for time frame:  53
Giant component for time frame:  54
Giant component for time frame:  55
Giant component for time frame:  56
Giant component for time frame:  57
Giant component for time frame:  58
Giant component for time frame:  59
Giant component for time frame:  60
Giant component for time frame:  61
Giant component for time frame:  62
Giant component for time frame:  63
Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time frame:  74
Giant component for time frame:  75
Giant component for time fra

Giant component for time frame:  272
Giant component for time frame:  273
Giant component for time frame:  274
Giant component for time frame:  275
Giant component for time frame:  276
Giant component for time frame:  277
Giant component for time frame:  278
Giant component for time frame:  279
Giant component for time frame:  280
Giant component for time frame:  281
Giant component for time frame:  282
Giant component for time frame:  283
Giant component for time frame:  284
Giant component for time frame:  285
Giant component for time frame:  286
Giant component for time frame:  287
Giant component for time frame:  288
Giant component for time frame:  289
Giant component for time frame:  290
Giant component for time frame:  291
Giant component for time frame:  292
Giant component for time frame:  293
Giant component for time frame:  294
Giant component for time frame:  295
Giant component for time frame:  296
Giant component for time frame:  297
Giant component for time frame:  298
G

Giant component for time frame:  47
Giant component for time frame:  48
Giant component for time frame:  49
Giant component for time frame:  50
Giant component for time frame:  51
Giant component for time frame:  52
Giant component for time frame:  53
Giant component for time frame:  54
Giant component for time frame:  55
Giant component for time frame:  56
Giant component for time frame:  57
Giant component for time frame:  58
Giant component for time frame:  59
Giant component for time frame:  60
Giant component for time frame:  61
Giant component for time frame:  62
Giant component for time frame:  63
Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time fra

...

Giant component for time frame:  165
Giant component for time frame:  166
Giant component for time frame:  167
Giant component for time frame:  168
Giant component for time frame:  169
Giant component for time frame:  170
Giant component for time frame:  171
Giant component for time frame:  172
Giant component for time frame:  173
Giant component for time frame:  174
Giant component for time frame:  175
Giant component for time frame:  176
Giant component for time frame:  177
Giant component for time frame:  178
Giant component for time frame:  179
Giant component for time frame:  180
Giant component for time frame:  181
Giant component for time frame:  182
Giant component for time frame:  183
Giant component for time frame:  184
Giant component for time frame:  185
Giant component for time frame:  186
Giant component for time frame:  187
Giant component for time frame:  188
Giant component for time frame:  189
Giant component for time frame:  190
Giant component for time frame:  191
G

Giant component for time frame:  387
Giant component for time frame:  388
Giant component for time frame:  389
Giant component for time frame:  390
Giant component for time frame:  391
Giant component for time frame:  392
Giant component for time frame:  393
Giant component for time frame:  394
Giant component for time frame:  395
Giant component for time frame:  396
Giant component for time frame:  397
Giant component for time frame:  398
Giant component for time frame:  399
Giant component for time frame:  400
Giant component for time frame:  401
Giant component for time frame:  402
Giant component for time frame:  403
Giant component for time frame:  404
Giant component for time frame:  405
Giant component for time frame:  406
Giant component for time frame:  407
Giant component for time frame:  1
Giant component for time frame:  2
Giant component for time frame:  3
Giant component for time frame:  4
Giant component for time frame:  5
Giant component for time frame:  6
Giant compone

Giant component for time frame:  205
Giant component for time frame:  206
Giant component for time frame:  207
Giant component for time frame:  208
Giant component for time frame:  209
Giant component for time frame:  210
Giant component for time frame:  211
Giant component for time frame:  212
Giant component for time frame:  213
Giant component for time frame:  214
Giant component for time frame:  215
Giant component for time frame:  216
Giant component for time frame:  217
Giant component for time frame:  218
Giant component for time frame:  219
Giant component for time frame:  220
Giant component for time frame:  221
Giant component for time frame:  222
Giant component for time frame:  223
Giant component for time frame:  224
Giant component for time frame:  225
Giant component for time frame:  226
Giant component for time frame:  227
Giant component for time frame:  228
Giant component for time frame:  229
Giant component for time frame:  230
Giant component for time frame:  231
G

Giant component for time frame:  64
Giant component for time frame:  65
Giant component for time frame:  66
Giant component for time frame:  67
Giant component for time frame:  68
Giant component for time frame:  69
Giant component for time frame:  70
Giant component for time frame:  71
Giant component for time frame:  72
Giant component for time frame:  73
Giant component for time frame:  74
Giant component for time frame:  75
Giant component for time frame:  76
Giant component for time frame:  77
Giant component for time frame:  78
Giant component for time frame:  79
Giant component for time frame:  80
Giant component for time frame:  81
Giant component for time frame:  82
Giant component for time frame:  83
Giant component for time frame:  84
Giant component for time frame:  85
Giant component for time frame:  86
Giant component for time frame:  87
Giant component for time frame:  88
Giant component for time frame:  89
Giant component for time frame:  90
Giant component for time fra

Giant component for time frame:  287
Giant component for time frame:  288
Giant component for time frame:  289
Giant component for time frame:  290
Giant component for time frame:  291
Giant component for time frame:  292
Giant component for time frame:  293
Giant component for time frame:  294
Giant component for time frame:  295
Giant component for time frame:  296
Giant component for time frame:  297
Giant component for time frame:  298
Giant component for time frame:  299
Giant component for time frame:  300
Giant component for time frame:  301
Giant component for time frame:  302
Giant component for time frame:  303
Giant component for time frame:  304
Giant component for time frame:  305
Giant component for time frame:  306
Giant component for time frame:  307
Giant component for time frame:  308
Giant component for time frame:  309
Giant component for time frame:  310
Giant component for time frame:  311
Giant component for time frame:  312
Giant component for time frame:  313
G

In [ ]:
#TO TEST IF NO MEMORY PROBLEMS - TODO TODO TODO
for subdir, dirs, files in sorted(os.walk(dir_fmri_desikan)):
    
    for file in files:
        #print(os.path.join(subdir, file))